In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import ast
import json
import random
import re
import warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, brier_score_loss
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
import torch
from torch.utils.data import DataLoader
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, EarlyStoppingCallback
warnings.filterwarnings('ignore')


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
else:
    pass
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
import transformers
if DEVICE == 'cuda':
    pass
else:
    pass


In [ ]:
DRIVE_DOWNLOADS = Path('/content/drive/MyDrive/Downloads')
RAGTRUTH_TRAIN_PATH = DRIVE_DOWNLOADS / 'Train.csv'
RAGTRUTH_TEST_PATH = DRIVE_DOWNLOADS / 'test.csv'
HALUBENCH_PATH = DRIVE_DOWNLOADS / 'halubench.csv'
TRIVIA_PATH = DRIVE_DOWNLOADS / 'triviaplus.csv'
OUTPUT_DIR = DRIVE_DOWNLOADS / 'rag_detector_outputs'
FIG_DIR = OUTPUT_DIR / 'figures'
MODEL_DIR = OUTPUT_DIR / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_any(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')
    else:
        pass
    ext = path.suffix.lower()
    if ext == '.csv':
        return pd.read_csv(path)
    else:
        pass
    if ext == '.parquet':
        return pd.read_parquet(path)
    else:
        pass
    if ext == '.jsonl':
        return pd.read_json(path, lines=True)
    else:
        pass
    if ext == '.json':
        try:
            obj = json.loads(path.read_text(encoding='utf-8'))
            if isinstance(obj, list):
                return pd.DataFrame(obj)
            else:
                pass
            if isinstance(obj, dict):
                for key in ['data', 'records', 'examples']:
                    if key in obj and isinstance(obj[key], list):
                        return pd.DataFrame(obj[key])
                    else:
                        pass
                else:
                    pass
                return pd.DataFrame(obj)
            else:
                pass
        except Exception:
            return pd.read_json(path, lines=True)
        else:
            pass
        finally:
            pass
    else:
        pass
    raise ValueError(f'Unsupported file type: {ext}')


In [ ]:
rag_train_raw = load_any(RAGTRUTH_TRAIN_PATH)
rag_test_raw = load_any(RAGTRUTH_TEST_PATH)
halu_raw = load_any(HALUBENCH_PATH)
trivia_raw = load_any(TRIVIA_PATH)


In [ ]:
def normalise_text(x):
    x = '' if pd.isna(x) else str(x)
    x = x.lower().strip()
    x = re.sub('\\s+', ' ', x)
    return x

def parse_ragtruth_label(x):
    if isinstance(x, dict):
        d = x
    else:
        try:
            d = ast.literal_eval(str(x))
        except Exception:
            d = {}
        else:
            pass
        finally:
            pass
    return int(bool(d.get('evident_conflict', 0)) or bool(d.get('baseless_info', 0)))

def metric_dict(y_true, y_pred, y_prob=None):
    result = {'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0), 'recall': recall_score(y_true, y_pred, zero_division=0), 'f1': f1_score(y_true, y_pred, zero_division=0), 'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0)}
    if y_prob is not None and len(np.unique(y_true)) > 1:
        result['auroc'] = roc_auc_score(y_true, y_prob)
        result['auprc'] = average_precision_score(y_true, y_prob)
        result['brier'] = brier_score_loss(y_true, y_prob)
    else:
        pass
    return result


In [ ]:
def build_ragtruth(df, split_name):
    return pd.DataFrame({'sample_id': df['id'].astype(str), 'question': df['query'].fillna('').astype(str), 'context': df['context'].fillna('').astype(str), 'answer': df['output'].fillna('').astype(str), 'label': df['hallucination_labels_processed'].apply(parse_ragtruth_label).astype(int), 'task_type': df['task_type'].astype(str), 'generator_model': df['model'].astype(str), 'temperature': df['temperature'].astype(float), 'dataset': f'RAGTruth_{split_name}'})


In [ ]:
def build_halubench(df):
    output = pd.DataFrame({'sample_id': df['id'].astype(str), 'question': df['question'].fillna('').astype(str), 'context': df['passage'].fillna('').astype(str), 'answer': df['answer'].fillna('').astype(str), 'label': df['label'].astype(str).str.upper().map({'PASS': 0, 'FAIL': 1}), 'source_ds': df['source_ds'].astype(str), 'dataset': 'HaluBench'})
    if output['label'].isna().any():
        raise ValueError('Unexpected HaluBench labels.')
    else:
        pass
    output['label'] = output['label'].astype(int)
    return output


In [ ]:
def build_trivia(df):
    return pd.DataFrame({'sample_id': np.arange(len(df)).astype(str), 'question': df['question'].fillna('').astype(str), 'context': df['article'].fillna('').astype(str), 'answer': df['answer'].fillna('').astype(str), 'label': df['response_level_label_binary'].astype(int), 'source_ds': df['source'].astype(str), 'split': df['split'].astype(str), 'generator_model': df['model'].astype(str), 'dataset': 'TRIVIA+'})


In [ ]:
rag_train = build_ragtruth(rag_train_raw, 'train')
rag_test = build_ragtruth(rag_test_raw, 'test')
halu = build_halubench(halu_raw)
trivia = build_trivia(trivia_raw)
trivia = trivia[trivia['answer'].str.len() > 0].reset_index(drop=True)


In [ ]:
def audit_dataset(name, df):
    pass
audit_dataset('RAGTruth train', rag_train)
audit_dataset('RAGTruth test', rag_test)
audit_dataset('HaluBench', halu)
audit_dataset('TRIVIA+', trivia)


In [ ]:
plot_df = pd.DataFrame({'RAGTruth train': rag_train['label'].value_counts().sort_index(), 'RAGTruth test': rag_test['label'].value_counts().sort_index(), 'HaluBench': halu['label'].value_counts().sort_index(), 'TRIVIA+': trivia['label'].value_counts().sort_index()}).T.fillna(0)
plot_df.columns = ['Non-hallucination (0)', 'Hallucination (1)']
ax = plot_df.plot(kind='bar', figsize=(10, 5))
ax.set_title('Class balance across datasets')
ax.set_xlabel('Dataset')
ax.set_ylabel('Number of examples')
ax.tick_params(axis='x', rotation=20)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '01_class_balance.png', dpi=300, bbox_inches='tight')


In [ ]:
rag_train['context_norm'] = rag_train['context'].map(normalise_text)
rag_test['context_norm'] = rag_test['context'].map(normalise_text)
train_contexts = set(rag_train['context_norm'])
test_overlap_mask = rag_test['context_norm'].isin(train_contexts)
overlap_count = int(test_overlap_mask.sum())
rag_test_disjoint = rag_test[~test_overlap_mask].copy().reset_index(drop=True)
rag_train.drop(columns=['context_norm'], inplace=True)
rag_test.drop(columns=['context_norm'], inplace=True)


In [ ]:
def make_key(df, columns):
    return df[columns].fillna('').astype(str).agg(' || '.join, axis=1).map(normalise_text)
rag_train_qca = make_key(rag_train, ['question', 'context', 'answer'])
rag_train_context = make_key(rag_train, ['context'])
halu_qca = make_key(halu, ['question', 'context', 'answer'])
halu_context = make_key(halu, ['context'])
halu['exact_qca_overlap'] = halu_qca.isin(set(rag_train_qca))
halu['exact_context_overlap'] = halu_context.isin(set(rag_train_context))
halu_eval = halu[~halu['exact_qca_overlap']].copy().reset_index(drop=True)
halu_eval_no_ragtruth = halu_eval[halu_eval['source_ds'].str.lower() != 'ragtruth'].copy().reset_index(drop=True)


In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
train_idx, val_idx = next(gss.split(rag_train, groups=rag_train['context'].map(normalise_text)))
rag_fit = rag_train.iloc[train_idx].copy().reset_index(drop=True)
rag_val = rag_train.iloc[val_idx].copy().reset_index(drop=True)
train_groups = set(rag_fit['context'].map(normalise_text))
val_groups = set(rag_val['context'].map(normalise_text))


In [ ]:
def build_text(df, mode='Q+E+A'):
    q = df['question'].fillna('').astype(str)
    e = df['context'].fillna('').astype(str)
    a = df['answer'].fillna('').astype(str)
    if mode == 'A':
        return 'Answer: ' + a
    else:
        pass
    if mode == 'Q+A':
        return 'Question: ' + q + '\nAnswer: ' + a
    else:
        pass
    if mode == 'E+A':
        return 'Evidence: ' + e + '\nAnswer: ' + a
    else:
        pass
    if mode == 'Q+E+A':
        return 'Question: ' + q + '\nEvidence: ' + e + '\nAnswer: ' + a
    else:
        pass
    raise ValueError(f'Unknown mode: {mode}')


In [ ]:
X_train = build_text(rag_fit, 'Q+E+A')
X_val = build_text(rag_val, 'Q+E+A')
y_train = rag_fit['label'].values
y_val = rag_val['label'].values
lr_model = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.98, sublinear_tf=True, max_features=250000)), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED))])
lr_model.fit(X_train, y_train)
lr_prob = lr_model.predict_proba(X_val)[:, 1]
lr_pred = (lr_prob >= 0.5).astype(int)
lr_result = {'model': 'TFIDF-LogReg', **metric_dict(y_val, lr_pred, lr_prob)}


In [ ]:
svm_model = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.98, sublinear_tf=True, max_features=250000)), ('clf', LinearSVC(class_weight='balanced', random_state=SEED))])
svm_model.fit(X_train, y_train)
svm_score = svm_model.decision_function(X_val)
svm_prob = 1 / (1 + np.exp(-np.clip(svm_score, -50, 50)))
svm_pred = (svm_score >= 0).astype(int)
svm_result = {'model': 'TFIDF-LinearSVM', **metric_dict(y_val, svm_pred, svm_prob)}
baseline_results = pd.DataFrame([lr_result, svm_result])
baseline_results.to_csv(OUTPUT_DIR / 'baseline_validation_results.csv', index=False)
joblib.dump(lr_model, MODEL_DIR / 'tfidf_logreg_qea.joblib')
joblib.dump(svm_model, MODEL_DIR / 'tfidf_svm_qea.joblib')


In [ ]:
TRANSFORMER_MODEL = 'microsoft/deberta-v3-base'
MAX_LEN = 512
gpu_memory_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 2) if torch.cuda.is_available() else 0
if gpu_memory_gb >= 20:
    TRAIN_BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16
    GRADIENT_ACCUMULATION_STEPS = 2
    USE_GRADIENT_CHECKPOINTING = False
elif gpu_memory_gb >= 14:
    TRAIN_BATCH_SIZE = 4
    EVAL_BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 4
    USE_GRADIENT_CHECKPOINTING = False
else:
    TRAIN_BATCH_SIZE = 2
    EVAL_BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 8
    USE_GRADIENT_CHECKPOINTING = False
LEARNING_RATE = 1e-05
WEIGHT_DECAY = 0.01
EPOCHS = 3
WARMUP_FRACTION = 0.1
MAX_GRAD_NORM = 1.0
QUESTION_BUDGET = 128
ANSWER_BUDGET = 128


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL, use_fast=True)


In [ ]:
train_texts_for_audit = build_text(rag_fit, 'Q+E+A').tolist()
val_texts_for_audit = build_text(rag_val, 'Q+E+A').tolist()

def get_true_token_lengths(texts, batch_size=32):
    lengths = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(batch_texts, truncation=False, padding=False, return_length=True)
        lengths.extend(encoded['length'])
    else:
        pass
    return np.array(lengths)
true_train_lengths = get_true_token_lengths(train_texts_for_audit)
true_val_lengths = get_true_token_lengths(val_texts_for_audit)
token_audit = pd.DataFrame({'split': ['train', 'validation'], 'n': [len(true_train_lengths), len(true_val_lengths)], 'mean_tokens': [true_train_lengths.mean(), true_val_lengths.mean()], 'median_tokens': [np.median(true_train_lengths), np.median(true_val_lengths)], 'max_tokens': [true_train_lengths.max(), true_val_lengths.max()], 'n_gt_512': [np.sum(true_train_lengths > MAX_LEN), np.sum(true_val_lengths > MAX_LEN)], 'pct_gt_512': [100 * np.mean(true_train_lengths > MAX_LEN), 100 * np.mean(true_val_lengths > MAX_LEN)]})
token_audit.to_csv(OUTPUT_DIR / 'ragtruth_true_token_length_audit.csv', index=False)


In [ ]:
from tqdm.auto import tqdm

def pack_transformer_example(question, evidence, answer, tokenizer, max_length=512, question_budget=128, answer_budget=128):
    question = '' if question is None else str(question)
    evidence = '' if evidence is None else str(evidence)
    answer = '' if answer is None else str(answer)
    question_ids = tokenizer('Question: ' + question, add_special_tokens=False, truncation=True, max_length=question_budget)['input_ids']
    answer_ids = tokenizer('Answer: ' + answer, add_special_tokens=False, truncation=True, max_length=answer_budget)['input_ids']
    remaining = max(0, max_length - len(question_ids) - len(answer_ids) - 2)
    evidence_ids = tokenizer(evidence, add_special_tokens=False, truncation=True, max_length=remaining)['input_ids']
    input_ids = [tokenizer.bos_token_id] + question_ids + answer_ids + evidence_ids + [tokenizer.eos_token_id]
    input_ids = input_ids[:max_length]
    return {'input_ids': input_ids, 'attention_mask': [1] * len(input_ids)}

def build_transformer_dataset(df, tokenizer, max_length=512, question_budget=128, answer_budget=128):
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Packing Transformer inputs'):
        packed = pack_transformer_example(question=row['question'], evidence=row['context'], answer=row['answer'], tokenizer=tokenizer, max_length=max_length, question_budget=question_budget, answer_budget=answer_budget)
        packed['labels'] = int(row['label'])
        records.append(packed)
    else:
        pass
    return HFDataset.from_list(records)
train_ds = build_transformer_dataset(rag_fit, tokenizer=tokenizer, max_length=MAX_LEN, question_budget=QUESTION_BUDGET, answer_budget=ANSWER_BUDGET)
val_ds = build_transformer_dataset(rag_val, tokenizer=tokenizer, max_length=MAX_LEN, question_budget=QUESTION_BUDGET, answer_budget=ANSWER_BUDGET)


In [ ]:
packed_train_lengths = np.array([len(ids) for ids in train_ds['input_ids']])
packed_val_lengths = np.array([len(ids) for ids in val_ds['input_ids']])


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
transformer_model = AutoModelForSequenceClassification.from_pretrained(TRANSFORMER_MODEL, num_labels=2, id2label={0: 'NON_HALLUCINATION', 1: 'HALLUCINATION'}, label2id={'NON_HALLUCINATION': 0, 'HALLUCINATION': 1})
transformer_model = transformer_model.to(DEVICE)
torch.cuda.empty_cache()


In [ ]:
if USE_GRADIENT_CHECKPOINTING:
    transformer_model.gradient_checkpointing_enable()
else:
    pass


In [ ]:
transformer_model.eval()
diagnostic_sample = train_ds.select(range(min(8, len(train_ds))))
diagnostic_inputs = {'input_ids': torch.tensor(diagnostic_sample['input_ids']).to(DEVICE), 'attention_mask': torch.tensor(diagnostic_sample['attention_mask']).to(DEVICE)}
if 'token_type_ids' in diagnostic_sample.column_names:
    diagnostic_inputs['token_type_ids'] = torch.tensor(diagnostic_sample['token_type_ids']).to(DEVICE)
else:
    pass
with torch.no_grad():
    diagnostic_outputs = transformer_model(**diagnostic_inputs)
diagnostic_logits = diagnostic_outputs.logits.detach().cpu().numpy()


In [ ]:
labels = np.array(train_ds['labels'])


In [ ]:
diagnostic_loader = DataLoader(train_ds, batch_size=2, shuffle=False, collate_fn=data_collator)
first_batch = next(iter(diagnostic_loader))
forward_kwargs = {k: v.to(DEVICE) for k, v in first_batch.items()}
with torch.no_grad():
    first_batch_output = transformer_model(**forward_kwargs)
first_batch_loss = first_batch_output.loss.item()


In [ ]:
for i in range(min(2, len(train_ds))):
    decoded = tokenizer.decode(train_ds[i]['input_ids'], skip_special_tokens=True)
else:
    pass


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).cpu().numpy()[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    return metric_dict(labels, predictions, probabilities)


In [ ]:
train_examples = len(train_ds)
steps_per_epoch = int(np.ceil(train_examples / (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)))
total_optimizer_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(total_optimizer_steps * WARMUP_FRACTION))


In [ ]:
training_args = TrainingArguments(output_dir=str(MODEL_DIR / 'deberta_qea'), learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, max_grad_norm=MAX_GRAD_NORM, warmup_steps=warmup_steps, lr_scheduler_type='linear', per_device_train_batch_size=TRAIN_BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, num_train_epochs=EPOCHS, eval_strategy='epoch', eval_on_start=True, save_strategy='epoch', save_total_limit=2, load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True, logging_strategy='steps', logging_steps=50, report_to='none', disable_tqdm=False, fp16=True, bf16=False, dataloader_num_workers=2, dataloader_pin_memory=True, seed=SEED)


In [ ]:
trainer = Trainer(model=transformer_model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
EFFECTIVE_BATCH_SIZE = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS


In [ ]:
transformer_model = AutoModelForSequenceClassification.from_pretrained(TRANSFORMER_MODEL, num_labels=2, id2label={0: 'NON_HALLUCINATION', 1: 'HALLUCINATION'}, label2id={'NON_HALLUCINATION': 0, 'HALLUCINATION': 1})
transformer_model = transformer_model.float()
transformer_model = transformer_model.to(DEVICE)
if USE_GRADIENT_CHECKPOINTING:
    transformer_model.gradient_checkpointing_enable()
else:
    pass


In [ ]:
transformer_model.eval()
diagnostic_sample = train_ds.select(range(min(8, len(train_ds))))
diagnostic_inputs = {'input_ids': torch.tensor(diagnostic_sample['input_ids'], device=DEVICE), 'attention_mask': torch.tensor(diagnostic_sample['attention_mask'], device=DEVICE)}
if 'token_type_ids' in diagnostic_sample.column_names:
    diagnostic_inputs['token_type_ids'] = torch.tensor(diagnostic_sample['token_type_ids'], device=DEVICE)
else:
    pass
with torch.no_grad():
    outputs = transformer_model(**diagnostic_inputs)


In [ ]:
EPOCHS = 3
LEARNING_RATE = 1e-05
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
WARMUP_FRACTION = 0.1
train_examples = len(train_ds)
steps_per_epoch = int(np.ceil(train_examples / (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)))
total_optimizer_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(total_optimizer_steps * WARMUP_FRACTION))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).cpu().numpy()[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    return metric_dict(labels, predictions, probabilities)
training_args = TrainingArguments(output_dir=str(MODEL_DIR / 'deberta_qea_fixed'), learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, max_grad_norm=MAX_GRAD_NORM, warmup_steps=warmup_steps, lr_scheduler_type='linear', per_device_train_batch_size=TRAIN_BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, num_train_epochs=EPOCHS, eval_strategy='epoch', eval_on_start=True, save_strategy='epoch', save_total_limit=2, load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True, logging_strategy='steps', logging_steps=50, report_to='none', disable_tqdm=False, fp16=True, bf16=False, dataloader_num_workers=2, dataloader_pin_memory=True, seed=SEED)
trainer = Trainer(model=transformer_model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, processing_class=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])


In [ ]:
from transformers import TrainingArguments, Trainer
smoke_args = TrainingArguments(output_dir='/content/drive/MyDrive/Downloads/rag_detector_outputs/smoke_test', max_steps=100, per_device_train_batch_size=4, per_device_eval_batch_size=4, gradient_accumulation_steps=4, learning_rate=2e-05, weight_decay=0.01, eval_strategy='steps', eval_steps=50, save_strategy='no', logging_steps=10, fp16=True, bf16=False, max_grad_norm=1.0, report_to='none')
smoke_trainer = Trainer(model=transformer_model, args=smoke_args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=data_collator, compute_metrics=compute_metrics)
result = smoke_trainer.train()


In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import Trainer
final_train_ds = train_ds
final_val_ds = val_ds
trainer = Trainer(model=transformer_model, args=training_args, train_dataset=final_train_ds, eval_dataset=final_val_ds, data_collator=data_collator, compute_metrics=compute_metrics)
final_train_result = trainer.train()


In [ ]:
BEST_MODEL_DIR = MODEL_DIR / 'deberta_qea_best'
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)


In [ ]:
pd.DataFrame([final_train_result.metrics]).to_csv(OUTPUT_DIR / 'deberta_training_metrics.csv', index=False)


In [ ]:
def transformer_predict(df, trainer_obj=trainer):
    dataset = build_transformer_dataset(df, tokenizer=tokenizer, max_length=MAX_LEN, question_budget=QUESTION_BUDGET, answer_budget=ANSWER_BUDGET)
    prediction_output = trainer_obj.predict(dataset)
    logits = prediction_output.predictions
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).cpu().numpy()[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    return (probabilities, predictions)


In [ ]:
def evaluate_transformer(dataset_name, df):
    probabilities, predictions = transformer_predict(df)
    metrics = metric_dict(df['label'].values, predictions, probabilities)
    result = {'model': 'DeBERTa-v3-base', 'dataset': dataset_name, 'n': len(df), **metrics}
    return (result, probabilities, predictions)


In [ ]:
ragtruth_result, ragtruth_prob, ragtruth_pred = evaluate_transformer('RAGTruth_test', rag_test)


In [ ]:
ragtruth_disjoint_result, ragtruth_disjoint_prob, ragtruth_disjoint_pred = evaluate_transformer('RAGTruth_test_context_disjoint', rag_test_disjoint)


In [ ]:
halubench_result, halubench_prob, halubench_pred = evaluate_transformer('HaluBench', halu_eval)


In [ ]:
halubench_no_ragtruth_result, halubench_no_ragtruth_prob, halubench_no_ragtruth_pred = evaluate_transformer('HaluBench_excluding_RAGTruth_source', halu_eval_no_ragtruth)


In [ ]:
trivia_test = trivia[trivia['split'].str.lower() == 'test'].copy().reset_index(drop=True)
trivia_result, trivia_prob, trivia_pred = evaluate_transformer('TRIVIA+_test', trivia_test)


In [ ]:
transformer_results = pd.DataFrame([ragtruth_result, ragtruth_disjoint_result, halubench_result, halubench_no_ragtruth_result, trivia_result])
transformer_results.to_csv(OUTPUT_DIR / 'deberta_all_dataset_results.csv', index=False)


In [ ]:
classical_rows = []
lr_test_prob = lr_model.predict_proba(build_text(rag_test, 'Q+E+A'))[:, 1]
lr_test_pred = (lr_test_prob >= 0.5).astype(int)
classical_rows.append({'model': 'TFIDF-LogReg', 'dataset': 'RAGTruth_test', 'n': len(rag_test), **metric_dict(rag_test['label'].values, lr_test_pred, lr_test_prob)})
svm_test_score = svm_model.decision_function(build_text(rag_test, 'Q+E+A'))
svm_test_prob = 1 / (1 + np.exp(-np.clip(svm_test_score, -50, 50)))
svm_test_pred = (svm_test_score >= 0).astype(int)
classical_rows.append({'model': 'TFIDF-LinearSVM', 'dataset': 'RAGTruth_test', 'n': len(rag_test), **metric_dict(rag_test['label'].values, svm_test_pred, svm_test_prob)})
classical_results = pd.DataFrame(classical_rows)


In [ ]:
main_model_comparison = pd.concat([classical_results, transformer_results[transformer_results['dataset'] == 'RAGTruth_test']], ignore_index=True)
main_model_comparison.to_csv(OUTPUT_DIR / 'main_model_comparison_ragtruth_test.csv', index=False)


In [ ]:
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Non-hallucination', 'Hallucination'])
    ax.set_yticklabels(['Non-hallucination', 'Hallucination'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center')
        else:
            pass
    else:
        pass
    plt.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches='tight')
plot_confusion_matrix(rag_test['label'].values, ragtruth_pred, 'DeBERTa — RAGTruth Test', '02_ragtruth_confusion_matrix.png')
plot_confusion_matrix(halu_eval['label'].values, halubench_pred, 'DeBERTa — HaluBench', '03_halubench_confusion_matrix.png')
plot_confusion_matrix(trivia_test['label'].values, trivia_pred, 'DeBERTa — TRIVIA+ Test', '04_trivia_confusion_matrix.png')


In [ ]:
def plot_roc_pr(df, probabilities, dataset_name, filename_prefix):
    y_true = df['label'].values
    fpr, tpr, _ = roc_curve(y_true, probabilities)
    auroc = roc_auc_score(y_true, probabilities)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f'AUROC={auroc:.3f}')
    ax.plot([0, 1], [0, 1], '--')
    ax.set_title(f'ROC — {dataset_name}')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'{filename_prefix}_roc.png', dpi=300, bbox_inches='tight')
    precision, recall, _ = precision_recall_curve(y_true, probabilities)
    auprc = average_precision_score(y_true, probabilities)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(recall, precision, label=f'AUPRC={auprc:.3f}')
    ax.set_title(f'Precision-Recall — {dataset_name}')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'{filename_prefix}_pr.png', dpi=300, bbox_inches='tight')


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

def plot_roc_pr(df, probabilities, dataset_name, filename_prefix):
    y_true = df['label'].values
    fpr, tpr, _ = roc_curve(y_true, probabilities)
    auroc = roc_auc_score(y_true, probabilities)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f'AUROC={auroc:.3f}')
    ax.plot([0, 1], [0, 1], '--')
    ax.set_title(f'ROC — {dataset_name}')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'{filename_prefix}_roc.png', dpi=300, bbox_inches='tight')
    precision, recall, _ = precision_recall_curve(y_true, probabilities)
    auprc = average_precision_score(y_true, probabilities)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(recall, precision, label=f'AUPRC={auprc:.3f}')
    ax.set_title(f'Precision-Recall — {dataset_name}')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'{filename_prefix}_pr.png', dpi=300, bbox_inches='tight')


In [ ]:
plot_roc_pr(rag_test, ragtruth_prob, 'RAGTruth Test', '05_ragtruth')
plot_roc_pr(halu_eval, halubench_prob, 'HaluBench', '06_halubench')
plot_roc_pr(trivia_test, trivia_prob, 'TRIVIA+ Test', '07_trivia')


In [ ]:
def plot_calibration(df, probabilities, dataset_name, filename):
    y_true = df['label'].values
    fraction_positive, mean_predicted = calibration_curve(y_true, probabilities, n_bins=10, strategy='uniform')
    brier = brier_score_loss(y_true, probabilities)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(mean_predicted, fraction_positive, marker='o', label=f'DeBERTa (Brier={brier:.3f})')
    ax.plot([0, 1], [0, 1], '--', label='Perfect calibration')
    ax.set_title(f'Calibration — {dataset_name}')
    ax.set_xlabel('Mean predicted hallucination probability')
    ax.set_ylabel('Observed hallucination frequency')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches='tight')
    return brier


In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
ragtruth_brier = plot_calibration(rag_test, ragtruth_prob, 'RAGTruth Test', '08_ragtruth_calibration.png')
halubench_brier = plot_calibration(halu_eval, halubench_prob, 'HaluBench', '09_halubench_calibration.png')
trivia_brier = plot_calibration(trivia_test, trivia_prob, 'TRIVIA+ Test', '10_trivia_calibration.png')


In [ ]:
ragtruth_f1 = transformer_results.loc[transformer_results['dataset'] == 'RAGTruth_test', 'f1'].iloc[0]
cross_domain = transformer_results.copy()
cross_domain['absolute_f1_drop_vs_ragtruth'] = ragtruth_f1 - cross_domain['f1']
cross_domain['relative_f1_drop_percent'] = 100 * cross_domain['absolute_f1_drop_vs_ragtruth'] / max(ragtruth_f1, 1e-12)
cross_domain.to_csv(OUTPUT_DIR / 'cross_dataset_f1_drop.csv', index=False)


In [ ]:
ragtruth_predictions = rag_test.copy()
ragtruth_predictions['prob_hallucination'] = ragtruth_prob
ragtruth_predictions['prediction'] = ragtruth_pred
ragtruth_predictions.to_csv(OUTPUT_DIR / 'predictions_ragtruth_test.csv', index=False)


In [ ]:
halubench_predictions = halu_eval.copy()
halubench_predictions['prob_hallucination'] = halubench_prob
halubench_predictions['prediction'] = halubench_pred
halubench_predictions.to_csv(OUTPUT_DIR / 'predictions_halubench.csv', index=False)


In [ ]:
halubench_no_ragtruth_predictions = halu_eval_no_ragtruth.copy()
halubench_no_ragtruth_predictions['prob_hallucination'] = halubench_no_ragtruth_prob
halubench_no_ragtruth_predictions['prediction'] = halubench_no_ragtruth_pred
halubench_no_ragtruth_predictions.to_csv(OUTPUT_DIR / 'predictions_halubench_excluding_ragtruth_source.csv', index=False)


In [ ]:
trivia_predictions = trivia_test.copy()
trivia_predictions['prob_hallucination'] = trivia_prob
trivia_predictions['prediction'] = trivia_pred
trivia_predictions.to_csv(OUTPUT_DIR / 'predictions_trivia_test.csv', index=False)


In [ ]:
transformer_results.to_csv(OUTPUT_DIR / 'FINAL_deberta_results.csv', index=False)


In [ ]:
main_model_comparison.to_csv(OUTPUT_DIR / 'FINAL_ragtruth_model_comparison.csv', index=False)


In [ ]:
calibration_summary = pd.DataFrame([{'dataset': 'RAGTruth_test', 'brier': ragtruth_brier}, {'dataset': 'HaluBench', 'brier': halubench_brier}, {'dataset': 'TRIVIA+_test', 'brier': trivia_brier}])
calibration_summary.to_csv(OUTPUT_DIR / 'FINAL_calibration_summary.csv', index=False)


In [ ]:
dataset_summary = pd.DataFrame([{'dataset': 'RAGTruth_train', 'n': len(rag_train), 'positive_rate': rag_train['label'].mean()}, {'dataset': 'RAGTruth_test', 'n': len(rag_test), 'positive_rate': rag_test['label'].mean()}, {'dataset': 'RAGTruth_test_context_disjoint', 'n': len(rag_test_disjoint), 'positive_rate': rag_test_disjoint['label'].mean()}, {'dataset': 'HaluBench_exact_overlap_removed', 'n': len(halu_eval), 'positive_rate': halu_eval['label'].mean()}, {'dataset': 'HaluBench_excluding_RAGTruth_source', 'n': len(halu_eval_no_ragtruth), 'positive_rate': halu_eval_no_ragtruth['label'].mean()}, {'dataset': 'TRIVIA+_test', 'n': len(trivia_test), 'positive_rate': trivia_test['label'].mean()}])
dataset_summary.to_csv(OUTPUT_DIR / 'FINAL_dataset_summary.csv', index=False)


In [ ]:
pass


In [ ]:
for file in sorted(OUTPUT_DIR.glob('*.csv')):
    pass
else:
    pass
for file in sorted(FIG_DIR.glob('*.png')):
    pass
else:
    pass
for file in sorted(MODEL_DIR.glob('*')):
    pass
else:
    pass


In [ ]:
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import Dataset as HFDataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, brier_score_loss
from scipy.stats import wilcoxon


In [ ]:
SEED = 20261003
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
else:
    pass
MAX_LEN = 512
QUESTION_BUDGET = 128
ANSWER_BUDGET = 128
model = trainer.model
model.eval()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(DEVICE)


In [ ]:
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/content/drive/MyDrive/Downloads/rag_detector_outputs'
else:
    pass
OUTPUT_DIR = str(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)
EVIDENCE_DIR = os.path.join(OUTPUT_DIR, 'evidence_sensitivity')
os.makedirs(EVIDENCE_DIR, exist_ok=True)


In [ ]:
def pack_transformer_example_evidence(question, evidence, answer, tokenizer, max_length=512, question_budget=128, answer_budget=128):
    question = '' if question is None else str(question)
    evidence = '' if evidence is None else str(evidence)
    answer = '' if answer is None else str(answer)
    question_ids = tokenizer('Question: ' + question, add_special_tokens=False, truncation=True, max_length=question_budget)['input_ids']
    answer_ids = tokenizer('Answer: ' + answer, add_special_tokens=False, truncation=True, max_length=answer_budget)['input_ids']
    remaining = max(0, max_length - len(question_ids) - len(answer_ids) - 2)
    if remaining > 0:
        evidence_ids = tokenizer(evidence, add_special_tokens=False, truncation=True, max_length=remaining)['input_ids']
    else:
        evidence_ids = []
    input_ids = [tokenizer.bos_token_id] + question_ids + answer_ids + evidence_ids + [tokenizer.eos_token_id]
    input_ids = input_ids[:max_length]
    return {'input_ids': input_ids, 'attention_mask': [1] * len(input_ids)}


In [ ]:
base_df = rag_test.copy().reset_index(drop=True)
n = len(base_df)


In [ ]:
contexts = base_df['context'].astype(str).tolist()
substitution_contexts = []
for i in range(n):
    candidate = (i + 1) % n
    attempts = 0
    while candidate != i and contexts[candidate] == contexts[i] and (attempts < n):
        candidate = (candidate + 1) % n
        attempts += 1
    else:
        pass
    substitution_contexts.append(contexts[candidate])
else:
    pass


In [ ]:
distractor_contexts = []
for i in range(n):
    candidate = (i + 137) % n
    attempts = 0
    while candidate != i and contexts[candidate] == contexts[i] and (attempts < n):
        candidate = (candidate + 1) % n
        attempts += 1
    else:
        pass
    distractor_contexts.append(contexts[candidate])
else:
    pass
intervention_df = pd.DataFrame({'sample_id': base_df['sample_id'].astype(str), 'question': base_df['question'].astype(str), 'original_evidence': base_df['context'].astype(str), 'answer': base_df['answer'].astype(str), 'label': base_df['label'].astype(int), 'substituted_evidence': substitution_contexts, 'distractor_evidence': distractor_contexts})


In [ ]:
def build_intervention_dataset(df, evidence_column):
    records = []
    for _, row in df.iterrows():
        packed = pack_transformer_example_evidence(question=row['question'], evidence=row[evidence_column], answer=row['answer'], tokenizer=tokenizer, max_length=MAX_LEN, question_budget=QUESTION_BUDGET, answer_budget=ANSWER_BUDGET)
        packed['labels'] = int(row['label'])
        records.append(packed)
    else:
        pass
    return HFDataset.from_list(records)


In [ ]:
def predict_intervention_dataset(dataset, description):
    prediction_output = trainer.predict(dataset)
    logits = prediction_output.predictions
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    return (probabilities, predictions)


In [ ]:
original_dataset = build_intervention_dataset(intervention_df, 'original_evidence')
original_prob, original_pred = predict_intervention_dataset(original_dataset, 'ORIGINAL EVIDENCE')


In [ ]:
intervention_df['removed_evidence'] = ''
removed_dataset = build_intervention_dataset(intervention_df, 'removed_evidence')
removed_prob, removed_pred = predict_intervention_dataset(removed_dataset, 'EVIDENCE REMOVED')


In [ ]:
substituted_dataset = build_intervention_dataset(intervention_df, 'substituted_evidence')
substituted_prob, substituted_pred = predict_intervention_dataset(substituted_dataset, 'CROSS-EXAMPLE EVIDENCE SUBSTITUTION')


In [ ]:
intervention_df['original_plus_distractor'] = intervention_df['original_evidence'] + '\n\n[DISTRACTOR EVIDENCE]\n' + intervention_df['distractor_evidence']
distractor_dataset = build_intervention_dataset(intervention_df, 'original_plus_distractor')
distractor_prob, distractor_pred = predict_intervention_dataset(distractor_dataset, 'ORIGINAL + ONE DISTRACTOR')


In [ ]:
results_df = intervention_df[['sample_id', 'question', 'answer', 'label']].copy()
results_df['p_original'] = original_prob
results_df['p_removed'] = removed_prob
results_df['p_substituted'] = substituted_prob
results_df['p_distractor'] = distractor_prob
results_df['pred_original'] = original_pred
results_df['pred_removed'] = removed_pred
results_df['pred_substituted'] = substituted_pred
results_df['pred_distractor'] = distractor_pred


In [ ]:
results_df['esi_removed'] = np.abs(results_df['p_removed'] - results_df['p_original'])
results_df['esi_substituted'] = np.abs(results_df['p_substituted'] - results_df['p_original'])
results_df['esi_distractor'] = np.abs(results_df['p_distractor'] - results_df['p_original'])
results_df['delta_removed'] = results_df['p_removed'] - results_df['p_original']
results_df['delta_substituted'] = results_df['p_substituted'] - results_df['p_original']
results_df['delta_distractor'] = results_df['p_distractor'] - results_df['p_original']


In [ ]:
results_df['flip_removed'] = (results_df['pred_removed'] != results_df['pred_original']).astype(int)
results_df['flip_substituted'] = (results_df['pred_substituted'] != results_df['pred_original']).astype(int)
results_df['flip_distractor'] = (results_df['pred_distractor'] != results_df['pred_original']).astype(int)


In [ ]:
def calculate_metrics(y_true, probabilities, predictions):
    result = {'n': len(y_true), 'accuracy': accuracy_score(y_true, predictions), 'precision': precision_score(y_true, predictions, zero_division=0), 'recall': recall_score(y_true, predictions, zero_division=0), 'f1': f1_score(y_true, predictions, zero_division=0), 'macro_f1': f1_score(y_true, predictions, average='macro', zero_division=0), 'auroc': roc_auc_score(y_true, probabilities), 'auprc': average_precision_score(y_true, probabilities), 'brier': brier_score_loss(y_true, probabilities)}
    return result


In [ ]:
y_true = results_df['label'].values
evaluation_rows = []
for name, prob_col, pred_col in [('Original', 'p_original', 'pred_original'), ('Evidence removed', 'p_removed', 'pred_removed'), ('Cross-example substitution', 'p_substituted', 'pred_substituted'), ('Original + distractor', 'p_distractor', 'pred_distractor')]:
    metrics = calculate_metrics(y_true, results_df[prob_col].values, results_df[pred_col].values)
    metrics['intervention'] = name
    evaluation_rows.append(metrics)
else:
    pass
evaluation_summary = pd.DataFrame(evaluation_rows)[['intervention', 'n', 'accuracy', 'precision', 'recall', 'f1', 'macro_f1', 'auroc', 'auprc', 'brier']]


In [ ]:
def bootstrap_mean_ci(values, n_bootstrap=2000, confidence=0.95, seed=SEED):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    bootstrap_means = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        sample = rng.choice(values, size=len(values), replace=True)
        bootstrap_means[i] = np.mean(sample)
    else:
        pass
    alpha = 1 - confidence
    lower = np.quantile(bootstrap_means, alpha / 2)
    upper = np.quantile(bootstrap_means, 1 - alpha / 2)
    return (float(np.mean(values)), float(lower), float(upper))
esi_rows = []
for intervention_name, esi_col, delta_col, flip_col in [('Evidence removed', 'esi_removed', 'delta_removed', 'flip_removed'), ('Cross-example substitution', 'esi_substituted', 'delta_substituted', 'flip_substituted'), ('Original + distractor', 'esi_distractor', 'delta_distractor', 'flip_distractor')]:
    values = results_df[esi_col].values
    mean_esi, ci_low, ci_high = bootstrap_mean_ci(values)
    signed_delta = results_df[delta_col].values
    try:
        wilcoxon_result = wilcoxon(signed_delta, zero_method='wilcox', alternative='two-sided')
        wilcoxon_stat = float(wilcoxon_result.statistic)
        wilcoxon_p = float(wilcoxon_result.pvalue)
    except Exception:
        wilcoxon_stat = np.nan
        wilcoxon_p = np.nan
    else:
        pass
    finally:
        pass
    esi_rows.append({'intervention': intervention_name, 'mean_esi': mean_esi, 'esi_ci_lower': ci_low, 'esi_ci_upper': ci_high, 'median_esi': float(np.median(values)), 'mean_signed_delta': float(np.mean(signed_delta)), 'median_signed_delta': float(np.median(signed_delta)), 'flip_rate': float(results_df[flip_col].mean()), 'wilcoxon_statistic': wilcoxon_stat, 'wilcoxon_p': wilcoxon_p})
else:
    pass
esi_summary = pd.DataFrame(esi_rows)


In [ ]:
class_rows = []
for label_value, label_name in [(0, 'Non-hallucination'), (1, 'Hallucination')]:
    subset = results_df[results_df['label'] == label_value]
    for intervention_name, esi_col in [('Evidence removed', 'esi_removed'), ('Cross-example substitution', 'esi_substituted'), ('Original + distractor', 'esi_distractor')]:
        values = subset[esi_col].values
        mean_esi, ci_low, ci_high = bootstrap_mean_ci(values)
        class_rows.append({'true_class': label_name, 'intervention': intervention_name, 'n': len(values), 'mean_esi': mean_esi, 'esi_ci_lower': ci_low, 'esi_ci_upper': ci_high, 'median_esi': float(np.median(values))})
    else:
        pass
else:
    pass
esi_by_class = pd.DataFrame(class_rows)


In [ ]:
pass


In [ ]:
def save_confusion_matrix(y_true, predictions, title, filename):
    cm = confusion_matrix(y_true, predictions)
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Non-hallucination', 'Hallucination'])
    ax.set_yticklabels(['Non-hallucination', 'Hallucination'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center')
        else:
            pass
    else:
        pass
    plt.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.savefig(os.path.join(EVIDENCE_DIR, filename), dpi=300, bbox_inches='tight')
save_confusion_matrix(y_true, original_pred, 'DeBERTa — Original Evidence', '01_original_confusion_matrix.png')
save_confusion_matrix(y_true, removed_pred, 'DeBERTa — Evidence Removed', '02_removed_confusion_matrix.png')
save_confusion_matrix(y_true, substituted_pred, 'DeBERTa — Cross-example Evidence Substitution', '03_substitution_confusion_matrix.png')
save_confusion_matrix(y_true, distractor_pred, 'DeBERTa — Original + Distractor', '04_distractor_confusion_matrix.png')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([results_df['esi_removed'], results_df['esi_substituted'], results_df['esi_distractor']], labels=['Evidence removed', 'Evidence substituted', 'Distractor added'], showfliers=False)
ax.set_ylabel('Evidence Sensitivity Index (ESI)')
ax.set_title('Evidence sensitivity of DeBERTa')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(EVIDENCE_DIR, '05_esi_distribution.png'), dpi=300, bbox_inches='tight')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
comparisons = [('p_removed', 'Evidence removed'), ('p_substituted', 'Evidence substituted'), ('p_distractor', 'Distractor added')]
for ax, (column, title) in zip(axes, comparisons):
    ax.scatter(results_df['p_original'], results_df[column], alpha=0.25, s=12)
    ax.plot([0, 1], [0, 1], '--')
    ax.set_xlabel('Original hallucination probability')
    ax.set_ylabel('Intervention probability')
    ax.set_title(title)
else:
    pass
plt.tight_layout()
plt.savefig(os.path.join(EVIDENCE_DIR, '06_probability_shift.png'), dpi=300, bbox_inches='tight')


In [ ]:
mean_probabilities = pd.DataFrame({'Intervention': ['Original', 'Evidence removed', 'Evidence substituted', 'Original + distractor'], 'Mean probability': [results_df['p_original'].mean(), results_df['p_removed'].mean(), results_df['p_substituted'].mean(), results_df['p_distractor'].mean()]})
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(mean_probabilities['Intervention'], mean_probabilities['Mean probability'])
ax.set_ylabel('Mean predicted hallucination probability')
ax.set_title('Mean hallucination probability under evidence interventions')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(EVIDENCE_DIR, '07_mean_probability.png'), dpi=300, bbox_inches='tight')


In [ ]:
flip_rates = pd.DataFrame({'Intervention': ['Evidence removed', 'Evidence substituted', 'Original + distractor'], 'Flip rate': [results_df['flip_removed'].mean(), results_df['flip_substituted'].mean(), results_df['flip_distractor'].mean()]})
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(flip_rates['Intervention'], flip_rates['Flip rate'])
ax.set_ylabel('Prediction flip rate')
ax.set_title('Prediction changes caused by evidence interventions')
ax.set_ylim(0, min(1.0, max(0.05, flip_rates['Flip rate'].max() * 1.2)))
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(EVIDENCE_DIR, '08_prediction_flip_rate.png'), dpi=300, bbox_inches='tight')


In [ ]:
results_df.to_csv(os.path.join(EVIDENCE_DIR, 'evidence_intervention_per_example.csv'), index=False)
evaluation_summary.to_csv(os.path.join(EVIDENCE_DIR, 'evidence_intervention_performance.csv'), index=False)
esi_summary.to_csv(os.path.join(EVIDENCE_DIR, 'evidence_sensitivity_summary.csv'), index=False)
esi_by_class.to_csv(os.path.join(EVIDENCE_DIR, 'evidence_sensitivity_by_class.csv'), index=False)
mean_probabilities.to_csv(os.path.join(EVIDENCE_DIR, 'mean_probability_by_intervention.csv'), index=False)
flip_rates.to_csv(os.path.join(EVIDENCE_DIR, 'prediction_flip_rates.csv'), index=False)


In [ ]:
pass


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import Dataset as HFDataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, brier_score_loss
from scipy.stats import wilcoxon


In [ ]:
SEED = 20261003
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
else:
    pass


In [ ]:
model = trainer.model
model.eval()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(DEVICE)


In [ ]:
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/content/drive/MyDrive/Downloads/rag_detector_outputs'
else:
    pass
CONTEXT_DIR = os.path.join(OUTPUT_DIR, 'context_complexity')
os.makedirs(CONTEXT_DIR, exist_ok=True)


In [ ]:
MAX_LEN = 512
QUESTION_BUDGET = 128
ANSWER_BUDGET = 128
EVIDENCE_BUDGET = MAX_LEN - QUESTION_BUDGET - ANSWER_BUDGET - 2


In [ ]:
base_df = rag_test.copy().reset_index(drop=True)
required_columns = ['question', 'context', 'answer', 'label']
missing_columns = [c for c in required_columns if c not in base_df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')
else:
    pass
base_df['question'] = base_df['question'].fillna('').astype(str)
base_df['context'] = base_df['context'].fillna('').astype(str)
base_df['answer'] = base_df['answer'].fillna('').astype(str)
base_df['label'] = base_df['label'].astype(int)


In [ ]:
contexts = base_df['context'].tolist()
n = len(base_df)
distractors = []
for i in range(n):
    candidate = (i + 137) % n
    attempts = 0
    while candidate != i and contexts[candidate] == contexts[i] and (attempts < n):
        candidate = (candidate + 1) % n
        attempts += 1
    else:
        pass
    distractors.append(contexts[candidate])
else:
    pass
base_df['distractor'] = distractors


In [ ]:
qa_records = []
for _, row in base_df.iterrows():
    question_ids = tokenizer('Question: ' + row['question'], add_special_tokens=False, truncation=True, max_length=QUESTION_BUDGET)['input_ids']
    answer_ids = tokenizer('Answer: ' + row['answer'], add_special_tokens=False, truncation=True, max_length=ANSWER_BUDGET)['input_ids']
    qa_records.append({'question_ids': question_ids, 'answer_ids': answer_ids})
else:
    pass


In [ ]:
original_evidence_ids = []
distractor_evidence_ids = []
for i, row in base_df.iterrows():
    original_ids = tokenizer(row['context'], add_special_tokens=False)['input_ids']
    distractor_ids = tokenizer(row['distractor'], add_special_tokens=False)['input_ids']
    original_evidence_ids.append(original_ids)
    distractor_evidence_ids.append(distractor_ids)
else:
    pass


In [ ]:
conditions = {'0pct_irrelevant': 0.0, '25pct_irrelevant': 0.25, '50pct_irrelevant': 0.5, '75pct_irrelevant': 0.75, '100pct_irrelevant': 1.0}

def build_mixed_evidence_tokens(original_ids, distractor_ids, irrelevant_fraction, evidence_budget):
    relevant_fraction = 1.0 - irrelevant_fraction
    target_original = int(round(evidence_budget * relevant_fraction))
    target_distractor = evidence_budget - target_original
    original_part = original_ids[:target_original]
    distractor_part = distractor_ids[:target_distractor]
    combined = original_part + distractor_part
    if len(combined) < evidence_budget:
        remaining = evidence_budget - len(combined)
        if len(original_part) < target_original:
            extra_original = original_ids[len(original_part):]
            combined.extend(extra_original[:remaining])
        else:
            pass
        if len(combined) < evidence_budget:
            extra_distractor = distractor_ids[len(distractor_part):]
            combined.extend(extra_distractor[:evidence_budget - len(combined)])
        else:
            pass
    else:
        pass
    return combined[:evidence_budget]


In [ ]:
def create_packed_input(question_ids, answer_ids, evidence_ids):
    input_ids = [tokenizer.bos_token_id] + question_ids + answer_ids + evidence_ids + [tokenizer.eos_token_id]
    input_ids = input_ids[:MAX_LEN]
    return {'input_ids': input_ids, 'attention_mask': [1] * len(input_ids)}


In [ ]:
def build_condition_dataset(irrelevant_fraction):
    records = []
    for i, row in base_df.iterrows():
        evidence_ids = build_mixed_evidence_tokens(original_evidence_ids[i], distractor_evidence_ids[i], irrelevant_fraction, EVIDENCE_BUDGET)
        packed = create_packed_input(qa_records[i]['question_ids'], qa_records[i]['answer_ids'], evidence_ids)
        packed['labels'] = int(row['label'])
        records.append(packed)
    else:
        pass
    return HFDataset.from_list(records)


In [ ]:
def predict_condition(dataset, condition_name):
    prediction_output = trainer.predict(dataset)
    logits = prediction_output.predictions
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    return (probabilities, predictions)


In [ ]:
def expected_calibration_error(y_true, probabilities, n_bins=10):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    bin_rows = []
    for i in range(n_bins):
        lower = bin_edges[i]
        upper = bin_edges[i + 1]
        if i == n_bins - 1:
            mask = (probabilities >= lower) & (probabilities <= upper)
        else:
            mask = (probabilities >= lower) & (probabilities < upper)
        count = mask.sum()
        if count == 0:
            continue
        else:
            pass
        confidence = probabilities[mask].mean()
        observed = y_true[mask].mean()
        weight = count / len(y_true)
        ece += weight * abs(confidence - observed)
        bin_rows.append({'bin_lower': lower, 'bin_upper': upper, 'count': int(count), 'mean_confidence': confidence, 'observed_frequency': observed})
    else:
        pass
    return (float(ece), pd.DataFrame(bin_rows))


In [ ]:
def calculate_metrics(y_true, probabilities, predictions):
    ece, _ = expected_calibration_error(y_true, probabilities)
    return {'accuracy': accuracy_score(y_true, predictions), 'precision': precision_score(y_true, predictions, zero_division=0), 'recall': recall_score(y_true, predictions, zero_division=0), 'f1': f1_score(y_true, predictions, zero_division=0), 'macro_f1': f1_score(y_true, predictions, average='macro', zero_division=0), 'auroc': roc_auc_score(y_true, probabilities), 'auprc': average_precision_score(y_true, probabilities), 'brier': brier_score_loss(y_true, probabilities), 'ece': ece, 'mean_hallucination_probability': float(np.mean(probabilities))}


In [ ]:
all_probabilities = {}
all_predictions = {}
summary_rows = []
y_true = base_df['label'].to_numpy()
for condition_name, irrelevant_fraction in conditions.items():
    dataset = build_condition_dataset(irrelevant_fraction)
    probabilities, predictions = predict_condition(dataset, condition_name)
    all_probabilities[condition_name] = probabilities
    all_predictions[condition_name] = predictions
    metrics = calculate_metrics(y_true, probabilities, predictions)
    summary_rows.append({'condition': condition_name, 'irrelevant_fraction': irrelevant_fraction, 'relevant_fraction': 1.0 - irrelevant_fraction, 'n': len(y_true), **metrics})
else:
    pass
context_summary = pd.DataFrame(summary_rows)


In [ ]:
baseline_name = '0pct_irrelevant'
baseline_prob = all_probabilities[baseline_name]
baseline_pred = all_predictions[baseline_name]


In [ ]:
sensitivity_rows = []
for condition_name in conditions:
    probabilities = all_probabilities[condition_name]
    predictions = all_predictions[condition_name]
    esi = np.abs(probabilities - baseline_prob)
    delta = probabilities - baseline_prob
    flips = (predictions != baseline_pred).astype(int)
    try:
        wilcoxon_result = wilcoxon(delta, zero_method='wilcox', alternative='two-sided')
        wilcoxon_statistic = float(wilcoxon_result.statistic)
        wilcoxon_p = float(wilcoxon_result.pvalue)
    except Exception:
        wilcoxon_statistic = np.nan
        wilcoxon_p = np.nan
    else:
        pass
    finally:
        pass
    sensitivity_rows.append({'condition': condition_name, 'irrelevant_fraction': conditions[condition_name], 'mean_esi': float(np.mean(esi)), 'median_esi': float(np.median(esi)), 'mean_signed_delta': float(np.mean(delta)), 'median_signed_delta': float(np.median(delta)), 'flip_rate': float(np.mean(flips)), 'wilcoxon_statistic': wilcoxon_statistic, 'wilcoxon_p': wilcoxon_p})
else:
    pass
sensitivity_summary = pd.DataFrame(sensitivity_rows)


In [ ]:
class_rows = []
for label_value, label_name in [(0, 'Non-hallucination'), (1, 'Hallucination')]:
    mask = y_true == label_value
    for condition_name in conditions:
        if condition_name == baseline_name:
            continue
        else:
            pass
        probabilities = all_probabilities[condition_name]
        esi = np.abs(probabilities[mask] - baseline_prob[mask])
        class_rows.append({'true_class': label_name, 'condition': condition_name, 'n': int(mask.sum()), 'mean_esi': float(np.mean(esi)), 'median_esi': float(np.median(esi))})
    else:
        pass
else:
    pass
esi_by_class = pd.DataFrame(class_rows)


In [ ]:
per_example = pd.DataFrame({'sample_id': np.arange(len(base_df)), 'label': y_true, 'p_0pct_irrelevant': all_probabilities['0pct_irrelevant'], 'p_25pct_irrelevant': all_probabilities['25pct_irrelevant'], 'p_50pct_irrelevant': all_probabilities['50pct_irrelevant'], 'p_75pct_irrelevant': all_probabilities['75pct_irrelevant'], 'p_100pct_irrelevant': all_probabilities['100pct_irrelevant'], 'pred_0pct_irrelevant': all_predictions['0pct_irrelevant'], 'pred_25pct_irrelevant': all_predictions['25pct_irrelevant'], 'pred_50pct_irrelevant': all_predictions['50pct_irrelevant'], 'pred_75pct_irrelevant': all_predictions['75pct_irrelevant'], 'pred_100pct_irrelevant': all_predictions['100pct_irrelevant']})


In [ ]:
pass


In [ ]:
x = context_summary['irrelevant_fraction'] * 100
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, context_summary['auroc'], marker='o', label='AUROC')
ax.plot(x, context_summary['macro_f1'], marker='o', label='Macro-F1')
ax.plot(x, context_summary['auprc'], marker='o', label='AUPRC')
ax.set_xlabel('Irrelevant evidence (%)')
ax.set_ylabel('Score')
ax.set_title('Detector performance under increasing irrelevant evidence')
ax.set_xticks([0, 25, 50, 75, 100])
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(CONTEXT_DIR, '01_performance_vs_irrelevant_context.png'), dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sensitivity_summary['irrelevant_fraction'] * 100, sensitivity_summary['mean_esi'], marker='o')
ax.set_xlabel('Irrelevant evidence (%)')
ax.set_ylabel('Mean ESI relative to 0% condition')
ax.set_title('Evidence sensitivity under increasing irrelevant context')
ax.set_xticks([0, 25, 50, 75, 100])
plt.tight_layout()
plt.savefig(os.path.join(CONTEXT_DIR, '02_esi_vs_irrelevant_context.png'), dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sensitivity_summary['irrelevant_fraction'] * 100, sensitivity_summary['flip_rate'], marker='o')
ax.set_xlabel('Irrelevant evidence (%)')
ax.set_ylabel('Prediction flip rate')
ax.set_title('Prediction instability under increasing irrelevant evidence')
ax.set_xticks([0, 25, 50, 75, 100])
plt.tight_layout()
plt.savefig(os.path.join(CONTEXT_DIR, '03_flip_rate_vs_irrelevant_context.png'), dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(context_summary['irrelevant_fraction'] * 100, context_summary['mean_hallucination_probability'], marker='o')
ax.set_xlabel('Irrelevant evidence (%)')
ax.set_ylabel('Mean predicted hallucination probability')
ax.set_title('Predicted hallucination probability vs context composition')
ax.set_xticks([0, 25, 50, 75, 100])
plt.tight_layout()
plt.savefig(os.path.join(CONTEXT_DIR, '04_mean_probability_vs_irrelevant_context.png'), dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for condition_name in conditions:
    probabilities = all_probabilities[condition_name]
    ece, bins = expected_calibration_error(y_true, probabilities)
    if len(bins) == 0:
        continue
    else:
        pass
    ax.plot(bins['mean_confidence'], bins['observed_frequency'], marker='o', label=f'{condition_name} (ECE={ece:.3f})')
else:
    pass
ax.plot([0, 1], [0, 1], '--', label='Perfect calibration')
ax.set_xlabel('Mean predicted hallucination probability')
ax.set_ylabel('Observed hallucination frequency')
ax.set_title('Calibration under context complexity')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(CONTEXT_DIR, '05_calibration_vs_context_complexity.png'), dpi=300, bbox_inches='tight')


In [ ]:
sample_count = min(300, len(per_example))
rng = np.random.default_rng(SEED)
sample_indices = rng.choice(len(per_example), size=sample_count, replace=False)
condition_percentages = np.array([0, 25, 50, 75, 100])
probability_matrix = np.column_stack([per_example.loc[sample_indices, 'p_0pct_irrelevant'].values, per_example.loc[sample_indices, 'p_25pct_irrelevant'].values, per_example.loc[sample_indices, 'p_50pct_irrelevant'].values, per_example.loc[sample_indices, 'p_75pct_irrelevant'].values, per_example.loc[sample_indices, 'p_100pct_irrelevant'].values])
fig, ax = plt.subplots(figsize=(9, 5))
for row in probability_matrix:
    ax.plot(condition_percentages, row, alpha=0.06)
else:
    pass
ax.plot(condition_percentages, probability_matrix.mean(axis=0), marker='o', linewidth=3, label='Mean')
ax.set_xlabel('Irrelevant evidence (%)')
ax.set_ylabel('Predicted hallucination probability')
ax.set_title('Individual prediction trajectories under context complexity')
ax.set_xticks([0, 25, 50, 75, 100])
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(CONTEXT_DIR, '06_prediction_trajectories.png'), dpi=300, bbox_inches='tight')


In [ ]:
context_summary.to_csv(os.path.join(CONTEXT_DIR, 'context_complexity_performance.csv'), index=False)
sensitivity_summary.to_csv(os.path.join(CONTEXT_DIR, 'context_complexity_sensitivity.csv'), index=False)
esi_by_class.to_csv(os.path.join(CONTEXT_DIR, 'context_complexity_esi_by_class.csv'), index=False)
per_example.to_csv(os.path.join(CONTEXT_DIR, 'context_complexity_per_example.csv'), index=False)


In [ ]:
pass


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import Dataset as HFDataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix
from scipy.stats import wilcoxon


In [ ]:
SEED = 20261003
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
else:
    pass


In [ ]:
model = trainer.model
model.eval()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(DEVICE)


In [ ]:
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/content/drive/MyDrive/Downloads/rag_detector_outputs'
else:
    pass
ABLATION_DIR = os.path.join(OUTPUT_DIR, 'input_ablation')
os.makedirs(ABLATION_DIR, exist_ok=True)


In [ ]:
MAX_LEN = 512
QUESTION_BUDGET = 128
ANSWER_BUDGET = 128
EVIDENCE_BUDGET = 252


In [ ]:
base_df = rag_test.copy().reset_index(drop=True)
required_columns = ['question', 'context', 'answer', 'label']
missing_columns = [c for c in required_columns if c not in base_df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')
else:
    pass
for column in ['question', 'context', 'answer']:
    base_df[column] = base_df[column].fillna('').astype(str)
else:
    pass
base_df['label'] = base_df['label'].astype(int)


In [ ]:
question_ids_all = []
answer_ids_all = []
evidence_ids_all = []
for _, row in base_df.iterrows():
    question_ids = tokenizer('Question: ' + row['question'], add_special_tokens=False, truncation=True, max_length=QUESTION_BUDGET)['input_ids']
    answer_ids = tokenizer('Answer: ' + row['answer'], add_special_tokens=False, truncation=True, max_length=ANSWER_BUDGET)['input_ids']
    evidence_ids = tokenizer(row['context'], add_special_tokens=False, truncation=True, max_length=EVIDENCE_BUDGET)['input_ids']
    question_ids_all.append(question_ids)
    answer_ids_all.append(answer_ids)
    evidence_ids_all.append(evidence_ids)
else:
    pass


In [ ]:
def pack_ablation_input(question_ids=None, evidence_ids=None, answer_ids=None):
    if question_ids is None:
        question_ids = []
    else:
        pass
    if evidence_ids is None:
        evidence_ids = []
    else:
        pass
    if answer_ids is None:
        answer_ids = []
    else:
        pass
    input_ids = [tokenizer.bos_token_id] + question_ids + answer_ids + evidence_ids + [tokenizer.eos_token_id]
    input_ids = input_ids[:MAX_LEN]
    return {'input_ids': input_ids, 'attention_mask': [1] * len(input_ids)}


In [ ]:
def build_ablation_dataset(condition):
    records = []
    for i in range(len(base_df)):
        q = question_ids_all[i]
        e = evidence_ids_all[i]
        a = answer_ids_all[i]
        if condition == 'A':
            packed = pack_ablation_input(answer_ids=a)
        elif condition == 'Q+A':
            packed = pack_ablation_input(question_ids=q, answer_ids=a)
        elif condition == 'E+A':
            packed = pack_ablation_input(evidence_ids=e, answer_ids=a)
        elif condition == 'Q+E':
            packed = pack_ablation_input(question_ids=q, evidence_ids=e)
        elif condition == 'Q+E+A':
            packed = pack_ablation_input(question_ids=q, evidence_ids=e, answer_ids=a)
        else:
            raise ValueError(f'Unknown condition: {condition}')
        packed['labels'] = int(base_df.loc[i, 'label'])
        records.append(packed)
    else:
        pass
    return HFDataset.from_list(records)


In [ ]:
def calculate_ece(y_true, probabilities, n_bins=10):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    rows = []
    for i in range(n_bins):
        lower = edges[i]
        upper = edges[i + 1]
        if i == n_bins - 1:
            mask = (probabilities >= lower) & (probabilities <= upper)
        else:
            mask = (probabilities >= lower) & (probabilities < upper)
        count = mask.sum()
        if count == 0:
            continue
        else:
            pass
        confidence = probabilities[mask].mean()
        observed = y_true[mask].mean()
        weight = count / len(y_true)
        ece += weight * abs(confidence - observed)
        rows.append({'mean_confidence': confidence, 'observed_frequency': observed, 'count': int(count)})
    else:
        pass
    return (float(ece), pd.DataFrame(rows))


In [ ]:
def calculate_metrics(y_true, probabilities, predictions):
    ece, _ = calculate_ece(y_true, probabilities)
    return {'accuracy': accuracy_score(y_true, predictions), 'precision': precision_score(y_true, predictions, zero_division=0), 'recall': recall_score(y_true, predictions, zero_division=0), 'f1': f1_score(y_true, predictions, zero_division=0), 'macro_f1': f1_score(y_true, predictions, average='macro', zero_division=0), 'auroc': roc_auc_score(y_true, probabilities), 'auprc': average_precision_score(y_true, probabilities), 'brier': brier_score_loss(y_true, probabilities), 'ece': ece, 'mean_hallucination_probability': float(np.mean(probabilities))}


In [ ]:
conditions = ['A', 'Q+A', 'E+A', 'Q+E', 'Q+E+A']
all_probabilities = {}
all_predictions = {}
summary_rows = []
y_true = base_df['label'].to_numpy()
for condition in conditions:
    dataset = build_ablation_dataset(condition)
    output = trainer.predict(dataset)
    logits = output.predictions
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    all_probabilities[condition] = probabilities
    all_predictions[condition] = predictions
    metrics = calculate_metrics(y_true, probabilities, predictions)
    summary_rows.append({'condition': condition, 'n': len(y_true), **metrics})
else:
    pass
summary_df = pd.DataFrame(summary_rows)


In [ ]:
full_prob = all_probabilities['Q+E+A']
full_pred = all_predictions['Q+E+A']


In [ ]:
difference_rows = []
for condition in conditions:
    if condition == 'Q+E+A':
        continue
    else:
        pass
    probability = all_probabilities[condition]
    prediction = all_predictions[condition]
    delta = probability - full_prob
    absolute_delta = np.abs(delta)
    flip = (prediction != full_pred).astype(int)
    try:
        test = wilcoxon(delta, zero_method='wilcox', alternative='two-sided')
        statistic = float(test.statistic)
        p_value = float(test.pvalue)
    except Exception:
        statistic = np.nan
        p_value = np.nan
    else:
        pass
    finally:
        pass
    difference_rows.append({'condition': condition, 'mean_probability_difference': float(np.mean(delta)), 'mean_absolute_probability_difference': float(np.mean(absolute_delta)), 'median_absolute_probability_difference': float(np.median(absolute_delta)), 'prediction_flip_rate_vs_full': float(np.mean(flip)), 'wilcoxon_statistic': statistic, 'wilcoxon_p': p_value})
else:
    pass
difference_df = pd.DataFrame(difference_rows)


In [ ]:
def plot_confusion_matrix(y_true, predictions, title, filename):
    cm = confusion_matrix(y_true, predictions)
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Non-hallucination', 'Hallucination'])
    ax.set_yticklabels(['Non-hallucination', 'Hallucination'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center')
        else:
            pass
    else:
        pass
    plt.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.savefig(os.path.join(ABLATION_DIR, filename), dpi=300, bbox_inches='tight')
for condition in conditions:
    safe_name = condition.replace('+', '_')
    plot_confusion_matrix(y_true, all_predictions[condition], f'DeBERTa — {condition}', f'confusion_{safe_name}.png')
else:
    pass


In [ ]:
x = np.arange(len(summary_df))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width, summary_df['auroc'], width, label='AUROC')
ax.bar(x, summary_df['macro_f1'], width, label='Macro-F1')
ax.bar(x + width, summary_df['auprc'], width, label='AUPRC')
ax.set_xticks(x)
ax.set_xticklabels(summary_df['condition'])
ax.set_ylabel('Score')
ax.set_title('DeBERTa performance under input ablation')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(ABLATION_DIR, '01_ablation_performance.png'), dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for condition in conditions:
    probabilities = all_probabilities[condition]
    ece, bins = calculate_ece(y_true, probabilities)
    if len(bins) == 0:
        continue
    else:
        pass
    ax.plot(bins['mean_confidence'], bins['observed_frequency'], marker='o', label=f'{condition} (ECE={ece:.3f})')
else:
    pass
ax.plot([0, 1], [0, 1], '--', label='Perfect calibration')
ax.set_xlabel('Mean predicted hallucination probability')
ax.set_ylabel('Observed hallucination frequency')
ax.set_title('Calibration under input ablation')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(ABLATION_DIR, '02_ablation_calibration.png'), dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(summary_df['condition'], summary_df['mean_hallucination_probability'])
ax.set_xlabel('Input condition')
ax.set_ylabel('Mean predicted hallucination probability')
ax.set_title('Predicted hallucination probability under input ablation')
plt.tight_layout()
plt.savefig(os.path.join(ABLATION_DIR, '03_mean_probability.png'), dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot([all_probabilities[c] for c in conditions], labels=conditions, showfliers=False)
ax.set_xlabel('Input condition')
ax.set_ylabel('Predicted hallucination probability')
ax.set_title('Prediction probability distributions')
plt.tight_layout()
plt.savefig(os.path.join(ABLATION_DIR, '04_probability_distributions.png'), dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(difference_df['condition'], difference_df['mean_absolute_probability_difference'])
ax.set_xlabel('Ablated condition')
ax.set_ylabel('Mean absolute probability difference')
ax.set_title('Change in predictions relative to Q+E+A')
plt.tight_layout()
plt.savefig(os.path.join(ABLATION_DIR, '05_difference_from_full_input.png'), dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(difference_df['condition'], difference_df['prediction_flip_rate_vs_full'])
ax.set_xlabel('Ablated condition')
ax.set_ylabel('Prediction flip rate vs Q+E+A')
ax.set_title('Prediction changes caused by input ablation')
plt.tight_layout()
plt.savefig(os.path.join(ABLATION_DIR, '06_prediction_flip_rate.png'), dpi=300, bbox_inches='tight')


In [ ]:
per_example = pd.DataFrame({'sample_id': np.arange(len(base_df)), 'label': y_true, 'p_A': all_probabilities['A'], 'p_QA': all_probabilities['Q+A'], 'p_EA': all_probabilities['E+A'], 'p_QE': all_probabilities['Q+E'], 'p_QEA': all_probabilities['Q+E+A'], 'pred_A': all_predictions['A'], 'pred_QA': all_predictions['Q+A'], 'pred_EA': all_predictions['E+A'], 'pred_QE': all_predictions['Q+E'], 'pred_QEA': all_predictions['Q+E+A']})


In [ ]:
summary_df.to_csv(os.path.join(ABLATION_DIR, 'input_ablation_performance.csv'), index=False)
difference_df.to_csv(os.path.join(ABLATION_DIR, 'input_ablation_difference_from_full.csv'), index=False)
per_example.to_csv(os.path.join(ABLATION_DIR, 'input_ablation_per_example.csv'), index=False)


In [ ]:
pass


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, wilcoxon
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
try:
    from statsmodels.stats.contingency_tables import mcnemar
    HAS_MCNEMAR = True
except Exception:
    HAS_MCNEMAR = False
else:
    pass
finally:
    pass


In [ ]:
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/content/drive/MyDrive/Downloads/rag_detector_outputs'
else:
    pass
STATS_DIR = os.path.join(OUTPUT_DIR, 'final_statistical_analysis')
os.makedirs(STATS_DIR, exist_ok=True)


In [ ]:
SEED = 20261003
rng = np.random.default_rng(SEED)

def bootstrap_ci(values, statistic=np.mean, n_boot=5000, confidence=0.95):
    values = np.asarray(values, dtype=float)
    observed = float(statistic(values))
    boot = np.empty(n_boot)
    for i in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boot[i] = statistic(sample)
    else:
        pass
    alpha = 1 - confidence
    lower = float(np.quantile(boot, alpha / 2))
    upper = float(np.quantile(boot, 1 - alpha / 2))
    return (observed, lower, upper)

def paired_bootstrap_difference(x, y, n_boot=5000, confidence=0.95):
    """
    Mean paired difference: x - y
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    differences = x - y
    observed = float(np.mean(differences))
    boot = np.empty(n_boot)
    for i in range(n_boot):
        indices = rng.integers(0, len(differences), size=len(differences))
        boot[i] = np.mean(differences[indices])
    else:
        pass
    alpha = 1 - confidence
    lower = float(np.quantile(boot, alpha / 2))
    upper = float(np.quantile(boot, 1 - alpha / 2))
    return (observed, lower, upper)

def format_p(p):
    if pd.isna(p):
        return 'NA'
    else:
        pass
    if p == 0:
        return '<1e-300'
    else:
        pass
    if p < 1e-300:
        return '<1e-300'
    else:
        pass
    if p < 0.001:
        return f'{p:.2e}'
    else:
        pass
    return f'{p:.4f}'

def cliffs_delta(x, y):
    """
    Simple paired effect size using
    difference direction.
    """
    d = np.asarray(x) - np.asarray(y)
    return float(np.mean(np.sign(d)))


In [ ]:
context_perf_path = os.path.join(OUTPUT_DIR, 'context_complexity', 'context_complexity_performance.csv')
context_sens_path = os.path.join(OUTPUT_DIR, 'context_complexity', 'context_complexity_sensitivity.csv')
ablation_perf_path = os.path.join(OUTPUT_DIR, 'input_ablation', 'input_ablation_performance.csv')
ablation_diff_path = os.path.join(OUTPUT_DIR, 'input_ablation', 'input_ablation_difference_from_full.csv')
evidence_perf_path = os.path.join(OUTPUT_DIR, 'evidence_sensitivity', 'evidence_intervention_performance.csv')
evidence_sens_path = os.path.join(OUTPUT_DIR, 'evidence_sensitivity', 'evidence_sensitivity_summary.csv')
evidence_per_example_path = os.path.join(OUTPUT_DIR, 'evidence_sensitivity', 'evidence_intervention_per_example.csv')
if os.path.exists(context_perf_path):
    context_perf = pd.read_csv(context_perf_path)
else:
    context_perf = context_summary.copy()
if os.path.exists(context_sens_path):
    context_sens = pd.read_csv(context_sens_path)
else:
    context_sens = sensitivity_summary.copy()
if os.path.exists(ablation_perf_path):
    ablation_perf = pd.read_csv(ablation_perf_path)
else:
    ablation_perf = summary_df.copy()
if os.path.exists(ablation_diff_path):
    ablation_diff = pd.read_csv(ablation_diff_path)
else:
    ablation_diff = difference_df.copy()
if os.path.exists(evidence_perf_path):
    evidence_perf = pd.read_csv(evidence_perf_path)
else:
    evidence_perf = evaluation_summary.copy()
if os.path.exists(evidence_sens_path):
    evidence_sens = pd.read_csv(evidence_sens_path)
else:
    evidence_sens = esi_summary.copy()
if os.path.exists(evidence_per_example_path):
    evidence_examples = pd.read_csv(evidence_per_example_path)
else:
    evidence_examples = results_df.copy()


In [ ]:
rq3_rows = []
p_original = evidence_examples['p_original'].to_numpy()
for condition_name, column in [('Evidence removed', 'p_removed'), ('Evidence substituted', 'p_substituted'), ('Original + distractor', 'p_distractor')]:
    p_intervention = evidence_examples[column].to_numpy()
    delta = p_intervention - p_original
    abs_delta = np.abs(delta)
    observed_diff, ci_low, ci_high = paired_bootstrap_difference(p_intervention, p_original)
    try:
        test = wilcoxon(delta, zero_method='wilcox', alternative='two-sided')
        w_stat = float(test.statistic)
        p_value = float(test.pvalue)
    except Exception:
        w_stat = np.nan
        p_value = np.nan
    else:
        pass
    finally:
        pass
    flip_column = {'Evidence removed': 'flip_removed', 'Evidence substituted': 'flip_substituted', 'Original + distractor': 'flip_distractor'}[condition_name]
    flip_rate = float(evidence_examples[flip_column].mean())
    rq3_rows.append({'intervention': condition_name, 'mean_probability_change': observed_diff, 'ci_lower': ci_low, 'ci_upper': ci_high, 'mean_absolute_change': float(np.mean(abs_delta)), 'median_absolute_change': float(np.median(abs_delta)), 'prediction_flip_rate': flip_rate, 'wilcoxon_statistic': w_stat, 'p_value': p_value})
else:
    pass
rq3_stats = pd.DataFrame(rq3_rows)


In [ ]:
x = context_perf['irrelevant_fraction'].to_numpy()
trend_variables = {'AUROC': context_perf['auroc'].to_numpy(), 'AUPRC': context_perf['auprc'].to_numpy(), 'Macro-F1': context_perf['macro_f1'].to_numpy(), 'Brier': context_perf['brier'].to_numpy(), 'ECE': context_perf['ece'].to_numpy(), 'Mean hallucination probability': context_perf['mean_hallucination_probability'].to_numpy(), 'Prediction flip rate': context_sens['flip_rate'].to_numpy(), 'Mean ESI': context_sens['mean_esi'].to_numpy()}
trend_rows = []
for metric_name, values in trend_variables.items():
    rho, p_value = spearmanr(x, values)
    trend_rows.append({'metric': metric_name, 'spearman_rho': float(rho), 'p_value': float(p_value)})
else:
    pass
trend_results = pd.DataFrame(trend_rows)


In [ ]:
context_0 = context_sens[context_sens['condition'] == '0pct_irrelevant']
context_100 = context_sens[context_sens['condition'] == '100pct_irrelevant']
context_metric_change = []
for metric in ['auroc', 'auprc', 'macro_f1', 'brier', 'ece']:
    if metric not in context_perf.columns:
        continue
    else:
        pass
    value_0 = float(context_perf.loc[context_perf['condition'] == '0pct_irrelevant', metric].iloc[0])
    value_100 = float(context_perf.loc[context_perf['condition'] == '100pct_irrelevant', metric].iloc[0])
    context_metric_change.append({'metric': metric, 'value_0pct': value_0, 'value_100pct': value_100, 'absolute_change': value_100 - value_0, 'relative_change_percent': (value_100 - value_0) / abs(value_0) * 100})
else:
    pass
context_metric_change = pd.DataFrame(context_metric_change)


In [ ]:
candidate_cross_dataset = [os.path.join('/content/drive/MyDrive/Downloads', 'FINAL_deberta_results.csv'), os.path.join('/content/drive/MyDrive/Downloads', 'deberta_all_dataset_results.csv'), os.path.join(OUTPUT_DIR, 'FINAL_deberta_results.csv'), '/mnt/data/FINAL_deberta_results.csv']
cross_dataset = None
for path in candidate_cross_dataset:
    if os.path.exists(path):
        try:
            temp = pd.read_csv(path)
            if len(temp) > 0:
                cross_dataset = temp
                break
            else:
                pass
        except Exception:
            pass
        else:
            pass
        finally:
            pass
    else:
        pass
else:
    pass
if cross_dataset is not None:
    pass
else:
    pass


In [ ]:
ablation_examples_path = os.path.join(OUTPUT_DIR, 'input_ablation', 'input_ablation_per_example.csv')
if os.path.exists(ablation_examples_path):
    ablation_examples = pd.read_csv(ablation_examples_path)
else:
    ablation_examples = None
ablation_stats = []
if ablation_examples is not None:
    full = ablation_examples['p_QEA'].to_numpy()
    for condition in ['A', 'Q+A', 'E+A', 'Q+E']:
        p = ablation_examples[f"p_{condition.replace('+', '')}"].to_numpy()
        delta = p - full
        abs_delta = np.abs(delta)
        diff, ci_low, ci_high = paired_bootstrap_difference(p, full)
        try:
            test = wilcoxon(delta, zero_method='wilcox', alternative='two-sided')
            w_stat = float(test.statistic)
            p_value = float(test.pvalue)
        except Exception:
            w_stat = np.nan
            p_value = np.nan
        else:
            pass
        finally:
            pass
        prediction_column = f"pred_{condition.replace('+', '')}"
        full_prediction_column = 'pred_QEA'
        flip_rate = float((ablation_examples[prediction_column] != ablation_examples[full_prediction_column]).mean())
        ablation_stats.append({'condition': condition, 'mean_probability_difference': diff, 'ci_lower': ci_low, 'ci_upper': ci_high, 'mean_absolute_difference': float(np.mean(abs_delta)), 'prediction_flip_rate': flip_rate, 'wilcoxon_statistic': w_stat, 'p_value': p_value})
    else:
        pass
else:
    pass
ablation_stats = pd.DataFrame(ablation_stats)


In [ ]:
esi_ci_rows = []
for _, row in context_sens.iterrows():
    condition = row['condition']
    if condition == '0pct_irrelevant':
        continue
    else:
        pass
    frac = float(row['irrelevant_fraction'])
    per_example_context_path = os.path.join(OUTPUT_DIR, 'context_complexity', 'context_complexity_per_example.csv')
    if os.path.exists(per_example_context_path):
        context_examples = pd.read_csv(per_example_context_path)
        baseline = context_examples['p_0pct_irrelevant'].to_numpy()
        current = context_examples[f'p_{int(frac * 100)}pct_irrelevant'].to_numpy()
        esi = np.abs(current - baseline)
        mean_esi, ci_low, ci_high = bootstrap_ci(esi)
        esi_ci_rows.append({'condition': condition, 'irrelevant_percent': frac * 100, 'mean_esi': mean_esi, 'ci_lower': ci_low, 'ci_upper': ci_high})
    else:
        pass
else:
    pass
esi_ci = pd.DataFrame(esi_ci_rows)


In [ ]:
rq_summary = pd.DataFrame({'RQ': ['RQ1', 'RQ2', 'RQ3', 'RQ4'], 'Experiment': ['In-domain DeBERTa detection', 'Cross-dataset distribution shift', 'Controlled evidence interventions', 'Context-complexity stress test'], 'Primary evidence': ['RAGTruth test performance', 'HaluBench + TRIVIA+ degradation', 'Evidence removal/substitution/distractor', '0–100% irrelevant evidence sweep'], 'Main quantity': ['AUROC / Macro-F1', 'AUROC/F1 change', 'ESI / flip rate', 'Trend in AUROC/F1/ESI/ECE'], 'Status': ['Completed', 'Completed', 'Completed', 'Completed']})


In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 6))
x_percent = context_perf['irrelevant_fraction'] * 100
line1 = ax1.plot(x_percent, context_perf['auroc'], marker='o', label='AUROC')
line2 = ax1.plot(x_percent, context_perf['macro_f1'], marker='o', label='Macro-F1')
ax1.set_xlabel('Irrelevant evidence (%)')
ax1.set_ylabel('Detection performance')
ax2 = ax1.twinx()
line3 = ax2.plot(x_percent, context_sens['mean_esi'], marker='s', label='Mean ESI')
ax2.set_ylabel('Mean Evidence Sensitivity Index')
lines = line1 + line2 + line3
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc='best')
ax1.set_title('DeBERTa robustness and evidence sensitivity under context shift')
plt.tight_layout()
plt.savefig(os.path.join(STATS_DIR, 'final_context_robustness_figure.png'), dpi=300, bbox_inches='tight')


In [ ]:
paper_table = context_perf[['condition', 'auroc', 'auprc', 'macro_f1', 'brier', 'ece', 'mean_hallucination_probability']].copy()
paper_table['irrelevant_percent'] = paper_table['condition'].str.extract('(\\d+)')[0].astype(int)
paper_table = paper_table[['irrelevant_percent', 'condition', 'auroc', 'auprc', 'macro_f1', 'brier', 'ece', 'mean_hallucination_probability']].sort_values('irrelevant_percent')


In [ ]:
rq3_stats.to_csv(os.path.join(STATS_DIR, 'RQ3_evidence_intervention_statistics.csv'), index=False)
trend_results.to_csv(os.path.join(STATS_DIR, 'RQ4_context_trend_tests.csv'), index=False)
context_metric_change.to_csv(os.path.join(STATS_DIR, 'RQ4_0_vs_100_context_change.csv'), index=False)
ablation_stats.to_csv(os.path.join(STATS_DIR, 'input_ablation_statistics.csv'), index=False)
esi_ci.to_csv(os.path.join(STATS_DIR, 'context_ESI_bootstrap_CI.csv'), index=False)
rq_summary.to_csv(os.path.join(STATS_DIR, 'RQ1_RQ4_consolidated_summary.csv'), index=False)
paper_table.to_csv(os.path.join(STATS_DIR, 'final_context_complexity_paper_table.csv'), index=False)


In [ ]:
for filename in ['RQ3_evidence_intervention_statistics.csv', 'RQ4_context_trend_tests.csv', 'RQ4_0_vs_100_context_change.csv', 'input_ablation_statistics.csv', 'context_ESI_bootstrap_CI.csv', 'RQ1_RQ4_consolidated_summary.csv', 'final_context_complexity_paper_table.csv', 'final_context_robustness_figure.png']:
    pass
else:
    pass


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, brier_score_loss


In [ ]:
required_objects = ['trainer', 'tokenizer', 'build_transformer_dataset', 'rag_test', 'rag_test_disjoint', 'halu_eval', 'halu_eval_no_ragtruth', 'trivia_test']
missing = [name for name in required_objects if name not in globals()]
if missing:
    raise NameError('The following required objects are missing: ' + ', '.join(missing))
else:
    pass


In [ ]:
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/content/drive/MyDrive/Downloads/rag_detector_outputs'
else:
    pass
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
def canonical_metrics(y_true, probabilities, predictions):
    return {'accuracy': accuracy_score(y_true, predictions), 'precision': precision_score(y_true, predictions, zero_division=0), 'recall': recall_score(y_true, predictions, zero_division=0), 'f1': f1_score(y_true, predictions, zero_division=0), 'macro_f1': f1_score(y_true, predictions, average='macro', zero_division=0), 'auroc': roc_auc_score(y_true, probabilities), 'auprc': average_precision_score(y_true, probabilities), 'brier': brier_score_loss(y_true, probabilities)}


In [ ]:
def canonical_predict(df):
    dataset = build_transformer_dataset(df, tokenizer=tokenizer, max_length=MAX_LEN, question_budget=QUESTION_BUDGET, answer_budget=ANSWER_BUDGET)
    output = trainer.predict(dataset)
    logits = output.predictions
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).cpu().numpy()[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    return (probabilities, predictions)


In [ ]:
evaluation_sets = {'RAGTruth_test': rag_test, 'RAGTruth_test_context_disjoint': rag_test_disjoint, 'HaluBench': halu_eval, 'HaluBench_excluding_RAGTruth_source': halu_eval_no_ragtruth, 'TRIVIA+_test': trivia_test}


In [ ]:
canonical_results = []
canonical_predictions = {}
for dataset_name, df in evaluation_sets.items():
    probabilities, predictions = canonical_predict(df)
    metrics = canonical_metrics(df['label'].values, probabilities, predictions)
    result = {'model': 'DeBERTa-v3-base', 'dataset': dataset_name, 'n': len(df), 'hallucination_rate': float(df['label'].mean()), **metrics}
    canonical_results.append(result)
    canonical_predictions[dataset_name] = {'probability': probabilities, 'prediction': predictions}
else:
    pass


In [ ]:
canonical_results_df = pd.DataFrame(canonical_results)


In [ ]:
reference_row = canonical_results_df[canonical_results_df['dataset'] == 'RAGTruth_test'].iloc[0]
reference_auroc = float(reference_row['auroc'])
reference_f1 = float(reference_row['f1'])
reference_macro_f1 = float(reference_row['macro_f1'])
reference_auprc = float(reference_row['auprc'])
reference_brier = float(reference_row['brier'])
drop_rows = []
for _, row in canonical_results_df.iterrows():
    if row['dataset'] == 'RAGTruth_test':
        continue
    else:
        pass
    drop_rows.append({'dataset': row['dataset'], 'auroc': row['auroc'], 'auroc_drop': reference_auroc - row['auroc'], 'auroc_drop_percent': (reference_auroc - row['auroc']) / reference_auroc * 100, 'f1': row['f1'], 'f1_drop': reference_f1 - row['f1'], 'f1_drop_percent': (reference_f1 - row['f1']) / reference_f1 * 100, 'macro_f1': row['macro_f1'], 'macro_f1_drop': reference_macro_f1 - row['macro_f1'], 'auprc': row['auprc'], 'auprc_drop': reference_auprc - row['auprc'], 'brier': row['brier'], 'brier_change': row['brier'] - reference_brier})
else:
    pass
cross_dataset_drop = pd.DataFrame(drop_rows)


In [ ]:
for dataset_name, df in evaluation_sets.items():
    result_df = df.copy()
    result_df['prob_hallucination'] = canonical_predictions[dataset_name]['probability']
    result_df['prediction'] = canonical_predictions[dataset_name]['prediction']
    safe_name = dataset_name.replace('+', 'plus').replace(' ', '_')
    result_df.to_csv(os.path.join(OUTPUT_DIR, f'CANONICAL_predictions_{safe_name}.csv'), index=False)
else:
    pass


In [ ]:
canonical_results_df.to_csv(os.path.join(OUTPUT_DIR, 'CANONICAL_deberta_all_dataset_results.csv'), index=False)
cross_dataset_drop.to_csv(os.path.join(OUTPUT_DIR, 'CANONICAL_cross_dataset_performance_drop.csv'), index=False)


In [ ]:
pass


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
from scipy.stats import wilcoxon
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix, roc_curve, precision_recall_curve
from statsmodels.stats.contingency_tables import mcnemar


In [ ]:
required_objects = ['rag_fit', 'rag_test']
missing = [x for x in required_objects if x not in globals()]
if missing:
    raise NameError('Missing notebook objects: ' + ', '.join(missing))
else:
    pass


In [ ]:
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/content/drive/MyDrive/Downloads/rag_detector_outputs'
else:
    pass
RQ1_DIR = os.path.join(OUTPUT_DIR, 'RQ1_model_comparison')
os.makedirs(RQ1_DIR, exist_ok=True)


In [ ]:
def make_text(df):
    return 'Question: ' + df['question'].fillna('').astype(str) + '\nEvidence: ' + df['context'].fillna('').astype(str) + '\nAnswer: ' + df['answer'].fillna('').astype(str)
train_text = make_text(rag_fit)
test_text = make_text(rag_test)
y_train = rag_fit['label'].astype(int).to_numpy()
y_test = rag_test['label'].astype(int).to_numpy()


In [ ]:
vectorizer = TfidfVectorizer(lowercase=True, strip_accents='unicode', ngram_range=(1, 2), min_df=2, max_df=0.98, sublinear_tf=True, max_features=200000)
X_train = vectorizer.fit_transform(train_text)
X_test = vectorizer.transform(test_text)


In [ ]:
lr_model = LogisticRegression(C=1.0, max_iter=2000, solver='liblinear', random_state=20261003)
lr_model.fit(X_train, y_train)
lr_prob = lr_model.predict_proba(X_test)[:, 1]
lr_pred = (lr_prob >= 0.5).astype(int)


In [ ]:
svm_model = LinearSVC(C=1.0, max_iter=5000, random_state=20261003)
svm_model.fit(X_train, y_train)
svm_decision = svm_model.decision_function(X_test)
svm_pred = (svm_decision >= 0).astype(int)
svm_prob = expit(svm_decision)


In [ ]:
def calculate_metrics(y_true, predictions, probabilities):
    return {'accuracy': accuracy_score(y_true, predictions), 'precision': precision_score(y_true, predictions, zero_division=0), 'recall': recall_score(y_true, predictions, zero_division=0), 'f1': f1_score(y_true, predictions, zero_division=0), 'macro_f1': f1_score(y_true, predictions, average='macro', zero_division=0), 'auroc': roc_auc_score(y_true, probabilities), 'auprc': average_precision_score(y_true, probabilities), 'brier': brier_score_loss(y_true, probabilities)}


In [ ]:
lr_metrics = calculate_metrics(y_test, lr_pred, lr_prob)
svm_metrics = calculate_metrics(y_test, svm_pred, svm_prob)


In [ ]:
canonical_file = os.path.join(OUTPUT_DIR, 'CANONICAL_deberta_all_dataset_results.csv')
canonical_results = pd.read_csv(canonical_file)
deberta_row = canonical_results[canonical_results['dataset'] == 'RAGTruth_test'].iloc[0]


In [ ]:
comparison = pd.DataFrame([{'model': 'TF-IDF + Logistic Regression', **lr_metrics}, {'model': 'TF-IDF + Linear SVM', **svm_metrics}, {'model': 'DeBERTa-v3-base', 'accuracy': deberta_row['accuracy'], 'precision': deberta_row['precision'], 'recall': deberta_row['recall'], 'f1': deberta_row['f1'], 'macro_f1': deberta_row['macro_f1'], 'auroc': deberta_row['auroc'], 'auprc': deberta_row['auprc'], 'brier': deberta_row['brier']}])


In [ ]:
deberta_prediction_file = os.path.join(OUTPUT_DIR, 'CANONICAL_predictions_RAGTruth_test.csv')
deberta_predictions_df = pd.read_csv(deberta_prediction_file)
deberta_prob = deberta_predictions_df['prob_hallucination'].to_numpy()
deberta_pred = deberta_predictions_df['prediction'].to_numpy()


In [ ]:
prediction_comparison = pd.DataFrame({'label': y_test, 'lr_probability': lr_prob, 'lr_prediction': lr_pred, 'svm_score': svm_decision, 'svm_probability_like': svm_prob, 'svm_prediction': svm_pred, 'deberta_probability': deberta_prob, 'deberta_prediction': deberta_pred})
prediction_comparison.to_csv(os.path.join(RQ1_DIR, 'RQ1_per_example_predictions.csv'), index=False)


In [ ]:
def mcnemar_test(pred_a, pred_b):
    table = np.zeros((2, 2), dtype=int)
    for a, b in zip(pred_a, pred_b):
        table[int(a), int(b)] += 1
    else:
        pass
    result = mcnemar(table, exact=True)
    return (table, float(result.statistic), float(result.pvalue))
pair_results = []
for model_a, pred_a, model_b, pred_b in [('DeBERTa', deberta_pred, 'Logistic Regression', lr_pred), ('DeBERTa', deberta_pred, 'Linear SVM', svm_pred), ('Logistic Regression', lr_pred, 'Linear SVM', svm_pred)]:
    table, statistic, pvalue = mcnemar_test(pred_a, pred_b)
    pair_results.append({'model_a': model_a, 'model_b': model_b, 'b00': table[0, 0], 'b01': table[0, 1], 'b10': table[1, 0], 'b11': table[1, 1], 'mcnemar_statistic': statistic, 'p_value': pvalue})
else:
    pass
mcnemar_results = pd.DataFrame(pair_results)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, probabilities in [('TF-IDF + Logistic Regression', lr_prob), ('TF-IDF + Linear SVM', svm_prob), ('DeBERTa-v3-base', deberta_prob)]:
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    auc = roc_auc_score(y_test, probabilities)
    ax.plot(fpr, tpr, label=f'{name} (AUROC={auc:.3f})')
else:
    pass
ax.plot([0, 1], [0, 1], '--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('RQ1 — ROC comparison on RAGTruth test')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RQ1_DIR, 'RQ1_ROC_comparison.png'), dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, probabilities in [('TF-IDF + Logistic Regression', lr_prob), ('TF-IDF + Linear SVM', svm_prob), ('DeBERTa-v3-base', deberta_prob)]:
    precision, recall, _ = precision_recall_curve(y_test, probabilities)
    auprc = average_precision_score(y_test, probabilities)
    ax.plot(recall, precision, label=f'{name} (AUPRC={auprc:.3f})')
else:
    pass
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('RQ1 — Precision-Recall comparison on RAGTruth test')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RQ1_DIR, 'RQ1_PR_comparison.png'), dpi=300, bbox_inches='tight')


In [ ]:
plot_df = comparison.copy()
x = np.arange(len(plot_df))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width, plot_df['auroc'], width, label='AUROC')
ax.bar(x, plot_df['auprc'], width, label='AUPRC')
ax.bar(x + width, plot_df['macro_f1'], width, label='Macro-F1')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['model'], rotation=15)
ax.set_ylabel('Score')
ax.set_title('RQ1 — Model comparison on RAGTruth test')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RQ1_DIR, 'RQ1_model_comparison.png'), dpi=300, bbox_inches='tight')


In [ ]:
def save_cm(y_true, predictions, title, filename):
    cm = confusion_matrix(y_true, predictions)
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Non-hallucination', 'Hallucination'])
    ax.set_yticklabels(['Non-hallucination', 'Hallucination'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center')
        else:
            pass
    else:
        pass
    plt.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.savefig(os.path.join(RQ1_DIR, filename), dpi=300, bbox_inches='tight')
save_cm(y_test, lr_pred, 'TF-IDF + Logistic Regression', 'RQ1_CM_Logistic.png')
save_cm(y_test, svm_pred, 'TF-IDF + Linear SVM', 'RQ1_CM_SVM.png')
save_cm(y_test, deberta_pred, 'DeBERTa-v3-base', 'RQ1_CM_DeBERTa.png')


In [ ]:
comparison.to_csv(os.path.join(RQ1_DIR, 'RQ1_final_model_comparison.csv'), index=False)
mcnemar_results.to_csv(os.path.join(RQ1_DIR, 'RQ1_mcnemar_tests.csv'), index=False)


In [ ]:
pass


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, brier_score_loss


In [ ]:
required_objects = ['vectorizer', 'lr_model', 'svm_model', 'halu_eval', 'halu_eval_no_ragtruth', 'trivia_test']
missing = [x for x in required_objects if x not in globals()]
if missing:
    raise NameError('Missing notebook objects: ' + ', '.join(missing))
else:
    pass


In [ ]:
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = '/content/drive/MyDrive/Downloads/rag_detector_outputs'
else:
    pass
RQ2_DIR = os.path.join(OUTPUT_DIR, 'RQ2_distribution_shift')
os.makedirs(RQ2_DIR, exist_ok=True)


In [ ]:
def make_text(df):
    return 'Question: ' + df['question'].fillna('').astype(str) + '\nEvidence: ' + df['context'].fillna('').astype(str) + '\nAnswer: ' + df['answer'].fillna('').astype(str)


In [ ]:
def calculate_metrics(y_true, predictions, probabilities):
    return {'accuracy': accuracy_score(y_true, predictions), 'precision': precision_score(y_true, predictions, zero_division=0), 'recall': recall_score(y_true, predictions, zero_division=0), 'f1': f1_score(y_true, predictions, zero_division=0), 'macro_f1': f1_score(y_true, predictions, average='macro', zero_division=0), 'auroc': roc_auc_score(y_true, probabilities), 'auprc': average_precision_score(y_true, probabilities), 'brier': brier_score_loss(y_true, probabilities)}


In [ ]:
external_sets = {'HaluBench': halu_eval, 'HaluBench_excluding_RAGTruth_source': halu_eval_no_ragtruth, 'TRIVIA+_test': trivia_test}


In [ ]:
all_results = []
all_predictions = {}
for dataset_name, df in external_sets.items():
    texts = make_text(df)
    X_external = vectorizer.transform(texts)
    y_true = df['label'].astype(int).to_numpy()
    lr_prob = lr_model.predict_proba(X_external)[:, 1]
    lr_pred = (lr_prob >= 0.5).astype(int)
    lr_metrics = calculate_metrics(y_true, lr_pred, lr_prob)
    all_results.append({'model': 'TF-IDF + Logistic Regression', 'dataset': dataset_name, 'n': len(df), 'hallucination_rate': float(y_true.mean()), **lr_metrics})
    all_predictions['TF-IDF + Logistic Regression', dataset_name] = {'y_true': y_true, 'probability': lr_prob, 'prediction': lr_pred}
    svm_score = svm_model.decision_function(X_external)
    svm_pred = (svm_score >= 0).astype(int)
    svm_prob = expit(svm_score)
    svm_metrics = calculate_metrics(y_true, svm_pred, svm_prob)
    all_results.append({'model': 'TF-IDF + Linear SVM', 'dataset': dataset_name, 'n': len(df), 'hallucination_rate': float(y_true.mean()), **svm_metrics})
    all_predictions['TF-IDF + Linear SVM', dataset_name] = {'y_true': y_true, 'score': svm_score, 'probability_like': svm_prob, 'prediction': svm_pred}
else:
    pass


In [ ]:
canonical_file = os.path.join(OUTPUT_DIR, 'CANONICAL_deberta_all_dataset_results.csv')
canonical_df = pd.read_csv(canonical_file)
deberta_rows = canonical_df[canonical_df['dataset'].isin(['HaluBench', 'HaluBench_excluding_RAGTruth_source', 'TRIVIA+_test'])].copy()
deberta_rows['model'] = 'DeBERTa-v3-base'
deberta_rows = deberta_rows[['model', 'dataset', 'n', 'hallucination_rate', 'accuracy', 'precision', 'recall', 'f1', 'macro_f1', 'auroc', 'auprc', 'brier']]


In [ ]:
classical_results = pd.DataFrame(all_results)
combined_results = pd.concat([classical_results, deberta_rows], ignore_index=True)


In [ ]:
pass


In [ ]:
rq1_comparison_file = os.path.join(OUTPUT_DIR, 'RQ1_model_comparison', 'RQ1_final_model_comparison.csv')
if os.path.exists(rq1_comparison_file):
    rq1_df = pd.read_csv(rq1_comparison_file)
else:
    raise FileNotFoundError('RQ1_final_model_comparison.csv not found.')


In [ ]:
drop_rows = []
for model_name in ['TF-IDF + Logistic Regression', 'TF-IDF + Linear SVM', 'DeBERTa-v3-base']:
    ref_row = rq1_df[rq1_df['model'] == model_name]
    if len(ref_row) == 0:
        continue
    else:
        pass
    ref_row = ref_row.iloc[0]
    reference_auroc = float(ref_row['auroc'])
    reference_f1 = float(ref_row['f1'])
    reference_macro_f1 = float(ref_row['macro_f1'])
    reference_auprc = float(ref_row['auprc'])
    for dataset_name in ['HaluBench', 'HaluBench_excluding_RAGTruth_source', 'TRIVIA+_test']:
        current = combined_results[(combined_results['model'] == model_name) & (combined_results['dataset'] == dataset_name)]
        if len(current) == 0:
            continue
        else:
            pass
        current = current.iloc[0]
        current_auroc = float(current['auroc'])
        current_f1 = float(current['f1'])
        current_macro_f1 = float(current['macro_f1'])
        current_auprc = float(current['auprc'])
        drop_rows.append({'model': model_name, 'dataset': dataset_name, 'reference_auroc': reference_auroc, 'external_auroc': current_auroc, 'auroc_drop': reference_auroc - current_auroc, 'auroc_drop_percent': (reference_auroc - current_auroc) / reference_auroc * 100, 'reference_f1': reference_f1, 'external_f1': current_f1, 'f1_drop': reference_f1 - current_f1, 'f1_drop_percent': (reference_f1 - current_f1) / reference_f1 * 100, 'reference_macro_f1': reference_macro_f1, 'external_macro_f1': current_macro_f1, 'macro_f1_drop': reference_macro_f1 - current_macro_f1, 'reference_auprc': reference_auprc, 'external_auprc': current_auprc, 'auprc_drop': reference_auprc - current_auprc})
    else:
        pass
else:
    pass
shift_results = pd.DataFrame(drop_rows)


In [ ]:
pass


In [ ]:
plot_datasets = ['HaluBench', 'HaluBench_excluding_RAGTruth_source', 'TRIVIA+_test']
model_order = ['TF-IDF + Logistic Regression', 'TF-IDF + Linear SVM', 'DeBERTa-v3-base']
x = np.arange(len(plot_datasets))
width = 0.25
fig, ax = plt.subplots(figsize=(11, 6))
for i, model_name in enumerate(model_order):
    values = []
    for dataset_name in plot_datasets:
        row = combined_results[(combined_results['model'] == model_name) & (combined_results['dataset'] == dataset_name)]
        values.append(float(row['auroc'].iloc[0]))
    else:
        pass
    ax.bar(x + (i - 1) * width, values, width, label=model_name)
else:
    pass
ax.set_xticks(x)
ax.set_xticklabels(['HaluBench', 'HaluBench\n(excl. RAGTruth source)', 'TRIVIA+'])
ax.set_ylabel('AUROC')
ax.set_title('Distribution-shift AUROC across detectors')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RQ2_DIR, 'RQ2_AUROC_distribution_shift.png'), dpi=300, bbox_inches='tight')


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for i, model_name in enumerate(model_order):
    values = []
    for dataset_name in plot_datasets:
        row = combined_results[(combined_results['model'] == model_name) & (combined_results['dataset'] == dataset_name)]
        values.append(float(row['macro_f1'].iloc[0]))
    else:
        pass
    ax.bar(x + (i - 1) * width, values, width, label=model_name)
else:
    pass
ax.set_xticks(x)
ax.set_xticklabels(['HaluBench', 'HaluBench\n(excl. RAGTruth source)', 'TRIVIA+'])
ax.set_ylabel('Macro-F1')
ax.set_title('Distribution-shift Macro-F1 across detectors')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RQ2_DIR, 'RQ2_MacroF1_distribution_shift.png'), dpi=300, bbox_inches='tight')


In [ ]:
for dataset_name, df in external_sets.items():
    output_df = pd.DataFrame({'label': df['label'].astype(int)})
    output_df['lr_probability'] = all_predictions['TF-IDF + Logistic Regression', dataset_name]['probability']
    output_df['lr_prediction'] = all_predictions['TF-IDF + Logistic Regression', dataset_name]['prediction']
    output_df['svm_score'] = all_predictions['TF-IDF + Linear SVM', dataset_name]['score']
    output_df['svm_probability_like'] = all_predictions['TF-IDF + Linear SVM', dataset_name]['probability_like']
    output_df['svm_prediction'] = all_predictions['TF-IDF + Linear SVM', dataset_name]['prediction']
    safe_name = dataset_name.replace('+', 'plus').replace(' ', '_')
    output_df.to_csv(os.path.join(RQ2_DIR, f'predictions_classical_{safe_name}.csv'), index=False)
else:
    pass


In [ ]:
combined_results.to_csv(os.path.join(RQ2_DIR, 'RQ2_all_detectors_external_results.csv'), index=False)
shift_results.to_csv(os.path.join(RQ2_DIR, 'RQ2_distribution_shift_drop.csv'), index=False)


In [ ]:
pass


In [ ]:
pass
